# Mapping and Predicting Flash Floods in Egypt from Space
### Arab Youth Space Hackathon 2026 · 813 Challenge · Theme 4: Climate-Induced Disasters

This notebook has **two halves**:

| Half | Question it answers | Method |
|---|---|---|
| **A. Detect** (Sections 2-9) | *Where did the water go in the storm that already happened?* | Sentinel-1 radar + Sentinel-2 optical change detection, terrain and sea masks, fusion, validation |
| **B. Predict** (Section 10) | *Where will water go in the NEXT storm, before it happens?* | Flood-susceptibility machine learning trained on terrain, land cover and pre-storm imagery, tested on a **different location** |

**What changed from the first version (fixes for the issues found in review)**
- ✅ **Sea / coastal mask** so Red Sea sediment plumes are no longer counted as flooded land.
- ✅ **Numbers audit (Section 6):** a waterfall showing exactly how the raw detections become the final km², so the totals reconcile.
- ✅ **Uncertainty range from a real threshold sweep** (not a guess).
- ✅ **Per-sensor, stratified validation with confidence intervals** (Section 9): each sensor is checked against the image it actually used.
- ✅ **Test of the "temporal decorrelation" explanation** (Section 9d) instead of just asserting it.
- ✅ **Second-location test** (Section 9e) with thresholds unchanged.
- ✅ **New: flood susceptibility ML** (Section 10) with spatial cross-validation, a transfer test, a risk map, and a priority list of road crossings.

**How to run:** run top to bottom in Google Colab (about 15-30 minutes, mostly downloads). Look at the picture after each step. Thresholds live in **Section 3**; change them there, not in the code. Outputs are saved next to the notebook (GeoTIFFs, CSVs, PNGs, a JSON summary and a `results_for_slides.md` with the numbers to paste into your deck).

> **Honesty note:** the notebook measures its own accuracy and prints where it fails. Report those numbers as they come out; a lower honest number is worth more to judges than a high unverified one.

## 0. Reading guide (start here if any term is new)

**The two satellites, in one sentence each**
- **Sentinel-2 (optical)** is a camera: it sees colours and also "invisible" infrared light. Water absorbs infrared, so water looks dark and *wet* in the water index. Blocked by clouds.
- **Sentinel-1 (radar)** sends its own microwave signal and listens to the echo. Calm water reflects the signal away, so it looks **dark**. Works through clouds and at night.

**Words you will see**
| Term | Plain English |
|---|---|
| **MNDWI** (water index) | A number per pixel. Above 0 = probably water. We care about how much it *rises* after the storm. |
| **dB (decibels)** | The unit of radar brightness. Land is roughly -5 to -15 dB, calm water is below about -18 dB. More negative = darker. |
| **Speckle** | Radar's natural grain. We smooth it slightly so it isn't mistaken for change. |
| **Change detection** | Compare BEFORE vs AFTER. Only pixels that *changed* count, which avoids permanent water and dark sand. |
| **Terrain mask** | Ignore steep or high ground. Water can't pool on a hillside. |
| **Precision** | Of the pixels I called flood, what fraction really were? (low = many false alarms) |
| **Recall** | Of the really flooded pixels, what fraction did I find? (low = I missed a lot) |
| **F1 / IoU** | One-number summaries of precision + recall. Closer to 1.0 is better. |

**Colour code used on every map**
| Colour | Meaning |
|---|---|
| 🟥 **Red** | Flood detected by **both** satellites (highest confidence) |
| 🟧 **Orange** | Flood detected by only one satellite |
| 🟦 **Blue** (in Section 7) | Optical only |
| 🟪 **Purple** (in Section 7) | Radar only |
| ⬜ **White bar** | Scale bar (distance on the ground) |

**Three questions to judge any map in this notebook**
1. *Does the red/orange follow valleys, low ground and dry riverbeds, or does it cover hillsides and flat desert?* (the second pattern means false alarms)
2. *Do the BEFORE and AFTER pictures actually look different where I flagged flood?*
3. *If I nudge a threshold a little, does the answer change a lot?* (Section 7d shows this)

Look for the boxes that start with **👀 What to look for** under each figure.

**Extra terms for the prediction half (Section 10)**
| Term | Plain English |
|---|---|
| **Susceptibility** | "How likely is this spot to get flooded when it rains hard?" A *relative risk score*, not a forecast of a specific storm. |
| **Upstream area** | How much land drains *into* a pixel. Big upstream area = water from a large catchment funnels here (the key driver of flash floods). |
| **HAND** | Height Above Nearest Drainage: how many metres a pixel sits above the closest wadi channel. Low = water reaches it easily. |
| **Spatial cross-validation** | Test the model on map blocks it never saw during training. (A random split would cheat, because neighbouring pixels are almost identical.) |
| **AUC / AP** | AUC: chance that a flooded pixel scores higher than a dry one (0.5 = coin flip, 1.0 = perfect). AP (average precision): like precision, but ranks-aware; compare it with the *base rate*, because floods are rare. |
| **Gain curve** | "If I flag the riskiest X% of the land, what share of the real flood do I catch?" The most intuitive result for decision-makers. |

## 1. Setup

In [ ]:
%pip install -q pystac-client planetary-computer odc-stac rioxarray xarray dask scipy scikit-image scikit-learn geopandas shapely rasterio pyproj matplotlib pandas osmnx folium pillow numba requests

In [ ]:
import warnings, json, os
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import pystac_client, planetary_computer as pc
from odc.stac import load as odc_load
from scipy import ndimage as ndi

catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=pc.sign_inplace,
)
print("Connected to Planetary Computer STAC.")

### 1b. Helper functions: detection maths (no internet needed)

In [ ]:
def to_db(x):
    x = np.asarray(x, dtype="float32")
    return 10 * np.log10(np.where(x > 0, x, np.nan))

def despeckle_db(db, size=3):
    # radar images are grainy ("speckle"); a small median filter calms it down
    m = ~np.isfinite(db)
    if m.all():
        return db
    fill = np.where(m, np.nanmedian(db), db)
    out = ndi.median_filter(fill, size=size)
    out[m] = np.nan
    return out

def reflectance(values, offset_dn=0):
    # Sentinel-2 L2A: DN -> reflectance. Newer scenes (processing baseline >= 04.00) carry a +1000 offset.
    arr = np.asarray(values, dtype="float32")
    arr = np.where(arr == 0, np.nan, arr)             # 0 = no data
    if np.isfinite(arr).any() and np.nanmax(arr) <= 3:  # already scaled by the loader
        return arr
    return np.clip((arr - offset_dn) / 10000.0, -0.1, 1.5)

def norm_diff(a, b):
    return (a - b) / (a + b + 1e-6)

def clean_mask(mask, min_pixels=10):
    # drop tiny specks (noise) from a boolean mask
    lab, n = ndi.label(mask)
    if n == 0:
        return mask.copy()
    sizes = np.bincount(lab.ravel())
    keep = sizes >= min_pixels
    keep[0] = False
    return keep[lab]

def slope_deg(dem, res):
    d = np.where(np.isfinite(dem), dem, np.nanmedian(dem))
    dzdy, dzdx = np.gradient(d, res)
    return np.degrees(np.arctan(np.hypot(dzdx, dzdy)))

def rel_elevation(dem, res, radius_m=1500):
    # crude "height above nearest low ground" (a simple stand-in for the HAND index)
    size = max(3, int(2 * radius_m / res) + 1)
    d = np.where(np.isfinite(dem), dem, np.nanmax(dem))
    return dem - ndi.minimum_filter(d, size=size)

def optical_flood(mndwi_pre, mndwi_post, d_thr=0.15, post_thr=0.0, pre_max=0.0):
    valid = np.isfinite(mndwi_pre) & np.isfinite(mndwi_post)
    d = mndwi_post - mndwi_pre
    return valid & (d > d_thr) & (mndwi_post > post_thr) & (mndwi_pre < pre_max)

def radar_flood(vv_pre_db, vv_post_db, post_thr=-18.0, diff_thr=-3.0, pre_min=-17.0):
    diff = vv_post_db - vv_pre_db
    return (np.isfinite(diff) & (vv_post_db < post_thr) & (diff < diff_thr) & (vv_pre_db > pre_min))

def weighted_confusion(pred, label, n_pred1, n_pred0):
    # accuracy from a sample stratified by predicted class (half predicted-flood, half predicted-dry),
    # re-weighted back to the full map so precision/recall are not distorted by the sampling.
    pred = np.asarray(pred).astype(int); label = np.asarray(label).astype(int)
    n1, n0 = (pred == 1).sum(), (pred == 0).sum()
    w1 = n_pred1 / n1 if n1 else 0.0
    w0 = n_pred0 / n0 if n0 else 0.0
    TP = w1 * ((pred == 1) & (label == 1)).sum(); FP = w1 * ((pred == 1) & (label == 0)).sum()
    FN = w0 * ((pred == 0) & (label == 1)).sum(); TN = w0 * ((pred == 0) & (label == 0)).sum()
    div = lambda a, b: a / b if b else np.nan
    prec, rec = div(TP, TP + FP), div(TP, TP + FN)
    return dict(precision=prec, recall=rec, f1=div(2 * prec * rec, prec + rec) if np.isfinite(prec) and np.isfinite(rec) else np.nan,
                iou=div(TP, TP + FP + FN), n_labeled=len(pred), TP=TP, FP=FP, FN=FN, TN=TN)

def utm_epsg(bbox):
    lon = (bbox[0] + bbox[2]) / 2; lat = (bbox[1] + bbox[3]) / 2
    return (32600 if lat >= 0 else 32700) + int((lon + 180) // 6) + 1

def top_components(mask, n=3, pad=0.35, min_size=60):
    # the n biggest connected flooded blobs, each with a square window around it (for zoom-in pictures)
    lab, k = ndi.label(mask)
    if k == 0:
        return []
    sizes = np.bincount(lab.ravel()); sizes[0] = 0
    objs = ndi.find_objects(lab)
    H, W = mask.shape
    out = []
    for idx in np.argsort(sizes)[::-1][:n]:
        if sizes[idx] == 0:
            continue
        sl = objs[idx - 1]
        r0, r1, c0, c1 = sl[0].start, sl[0].stop, sl[1].start, sl[1].stop
        half = int(max(r1 - r0, c1 - c0, min_size) * (0.5 + pad))
        rc, cc = (r0 + r1) // 2, (c0 + c1) // 2
        out.append((int(sizes[idx]), (max(0, rc - half), min(H, rc + half), max(0, cc - half), min(W, cc + half))))
    return out

def diagnose(m):
    # turn numbers into plain-English traffic lights. Returns [(level, message)], level in {"ok","warn","bad"}
    out = []
    if m["fused_km2"] <= 0:
        out.append(("bad", "No flooded area found. Check the event date/box (Section 4), or loosen the thresholds in Section 3."))
        return out
    f = m["flooded_frac"]
    if f > 0.25:
        out.append(("warn", f"{f*100:.0f}% of the whole box is flagged as flood. That is a lot for a desert. Suspect sand or shadow false alarms: tighten thresholds or terrain mask."))
    else:
        out.append(("ok", f"Flooded share of the box is {f*100:.1f}%, a plausible magnitude for a flash flood."))
    if m.get("agree_iou") is not None:
        a = m["agree_iou"]
        if a >= 0.4: out.append(("ok", f"Optical and radar overlap well (IoU {a:.2f}): two independent sensors support the result."))
        elif a >= 0.15: out.append(("warn", f"Optical and radar only partly overlap (IoU {a:.2f}). Normal when one sensor sees mud and the other water, but look at the zoom-ins."))
        else: out.append(("warn", f"Optical and radar barely overlap (IoU {a:.2f}). Treat red (both) pixels as the reliable core and the rest as candidates."))
    if m.get("terrain_removed_frac") is not None:
        t = m["terrain_removed_frac"]
        if t > 0.5: out.append(("warn", f"The terrain mask removed {t*100:.0f}% of raw detections. The raw detectors produce many impossible hits here (likely shadows/sand), so the final map leans heavily on the mask."))
        else: out.append(("ok", f"The terrain mask removed {t*100:.0f}% of raw detections, a modest clean-up."))
    if m.get("sens_change") is not None:
        c = m["sens_change"]
        if c > 0.5: out.append(("warn", f"The flooded area changes by {c*100:.0f}% if thresholds move ±20%. The area estimate is fragile: report it as a range."))
        else: out.append(("ok", f"Moving thresholds ±20% changes the area by {c*100:.0f}%. Reasonably stable."))
    d = m.get("days_to_first_after")
    if d is not None:
        if d > 7: out.append(("warn", f"The first clear AFTER image is {d} days after the storm. Flash-flood water in deserts can vanish in days, so this may under-count the peak."))
        else: out.append(("ok", f"The first AFTER image is only {d} day(s) after the storm, so the water is probably still there."))
    cf = m.get("clear_frac_post")
    if cf is not None and cf < 0.5:
        out.append(("warn", f"Only {cf*100:.0f}% of the best AFTER optical image is cloud-free; radar carries more of the weight."))
    if m.get("f1") is not None:
        out.append(("ok" if m["f1"] >= 0.7 else "warn", f"Against your hand labels F1 = {m['f1']:.2f}" + (" (good)." if m["f1"] >= 0.7 else " (below 0.7: improve thresholds or say so honestly in the limitations).")))
    return out

### 1c. Helper functions: terrain, ML and validation maths (no internet needed)

In [ ]:
from pyproj import Transformer
from matplotlib.colors import ListedColormap
from skimage.morphology import reconstruction
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance

# ---------- grid utilities ----------
def fit_shape(a, shape, fill=np.nan):
    # make sure two rasters loaded separately sit on the same grid (they should; this guards 1-pixel edge differences)
    a = np.asarray(a)
    if a.shape == tuple(shape):
        return a
    if abs(a.shape[0] - shape[0]) <= 2 and abs(a.shape[1] - shape[1]) <= 2:
        out = np.full(shape, fill if np.issubdtype(a.dtype, np.floating) else 0, dtype=a.dtype)
        h, w = min(a.shape[0], shape[0]), min(a.shape[1], shape[1])
        out[:h, :w] = a[:h, :w]
        return out
    raise ValueError(f"grid mismatch {a.shape} vs {shape}")

def iou(a, b):
    u = (a | b).sum()
    return float((a & b).sum() / u) if u else np.nan

# ---------- terrain analysis (hydrology) ----------
def fill_sinks(dem):
    # fills closed depressions so that water can always find a way downhill (morphological reconstruction)
    d = np.where(np.isfinite(dem), dem, np.nanmax(dem)).astype("float64")
    seed = d.copy(); seed[1:-1, 1:-1] = d.max()
    return reconstruction(seed, d, method="erosion")

_OFFS = [(-1, -1), (-1, 0), (-1, 1), (0, -1), (0, 1), (1, -1), (1, 0), (1, 1)]

def d8_receivers(z, res):
    # for every cell: flat index of its steepest downhill neighbour (-1 = none / sink)
    H, W = z.shape
    pad = np.pad(z, 1, mode="edge")
    best_drop = np.zeros((H, W)); best = np.full((H, W), -1, int)
    for k, (dy, dx) in enumerate(_OFFS):
        nb = pad[1 + dy:1 + dy + H, 1 + dx:1 + dx + W]
        drop = (z - nb) / (res * np.hypot(dy, dx))
        better = drop > best_drop
        best_drop = np.where(better, drop, best_drop); best = np.where(better, k, best)
    rr, cc = np.mgrid[0:H, 0:W]
    recv = np.full(H * W, -1, dtype=np.int64)
    for k, (dy, dx) in enumerate(_OFFS):
        m = (best == k)
        r2, c2 = rr[m] + dy, cc[m] + dx
        ok = (r2 >= 0) & (r2 < H) & (c2 >= 0) & (c2 < W)
        recv[np.flatnonzero(m.ravel())[ok]] = r2[ok] * W + c2[ok]
    return recv.reshape(H, W), best_drop

try:
    import numba
    @numba.njit(cache=False)
    def _acc_nb(order, recv, acc):
        for i in order:
            r = recv[i]
            if r >= 0:
                acc[r] += acc[i]
    def _accumulate(order, recv, acc):
        _acc_nb(order, recv, acc); return acc
except Exception:
    def _accumulate(order, recv, acc):
        a, rl = acc.tolist(), recv.tolist()
        for i in order.tolist():
            r = rl[i]
            if r >= 0:
                a[r] += a[i]
        return np.array(a)

def flow_accumulation(z, res):
    # number of upstream cells draining through each cell (D8 flow routing)
    zf = fill_sinks(z)
    zf = zf + np.random.default_rng(0).random(zf.shape) * 1e-4      # tiny tilt so flat areas drain
    recv, _ = d8_receivers(zf, res)
    order = np.argsort(zf.ravel())[::-1]                             # high -> low
    acc = _accumulate(order, recv.ravel(), np.ones(zf.size, dtype=np.float64))
    return acc.reshape(z.shape)

def terrain_features(dem, res, drain_km2=0.25):
    d = np.where(np.isfinite(dem), dem, np.nanmedian(dem)).astype("float64")
    d = ndi.gaussian_filter(d, 1.0)
    slope = slope_deg(d, res)
    acc = flow_accumulation(d, res)
    area_km2 = acc * (res / 1000.0) ** 2
    network = area_km2 >= drain_km2                                  # cells that behave like a wadi channel
    dist, idx = ndi.distance_transform_edt(~network, sampling=res, return_indices=True) if network.any() else (np.full(d.shape, np.nan), None)
    hand = np.maximum(d - d[idx[0], idx[1]], 0) if idx is not None else np.full(d.shape, np.nan)
    sca = np.maximum(acc * res, res)                                 # specific catchment area (m)
    twi = np.log(sca / np.tan(np.radians(np.maximum(slope, 0.5))))   # topographic wetness index
    curv = ndi.laplace(d) / (res ** 2)
    return dict(elev=d, slope=slope, log_area=np.log10(np.maximum(area_km2, 1e-6)), twi=twi, dist_drain=dist, hand=hand, curv=curv,
                rel500=rel_elevation(d, res, 500), rel1500=rel_elevation(d, res, 1500), network=network)

# ---------- ML utilities ----------
def balanced_w(y):
    y = np.asarray(y)
    w = np.where(y == 1, 0.5 / max((y == 1).sum(), 1), 0.5 / max((y == 0).sum(), 1))
    return w * len(y)

def make_model():
    return HistGradientBoostingClassifier(max_depth=6, learning_rate=0.08, max_iter=200, l2_regularization=1.0, early_stopping=False, random_state=0)

def block_folds(shape, res, block_m, k, seed=0):
    # assign every pixel to one of k folds, in square spatial blocks (so train/test pixels are never neighbours)
    rr, cc = np.mgrid[0:shape[0], 0:shape[1]]
    bid = (rr * res // block_m).astype(np.int64) * 100000 + (cc * res // block_m).astype(np.int64)
    uniq, inv = np.unique(bid, return_inverse=True)
    fold_of_block = np.random.default_rng(seed).integers(0, k, size=len(uniq))
    return fold_of_block[inv].reshape(shape)

def sample_xy(F, y, domain, folds, f_excl=None, max_pos=20000, neg_ratio=4, seed=0):
    rng = np.random.default_rng(seed)
    ff = folds.ravel(); dom = domain.ravel(); yy = y.ravel()
    ok = dom if f_excl is None else (dom & (ff != f_excl))
    pos = np.flatnonzero(ok & yy); neg = np.flatnonzero(ok & ~yy)
    if len(pos) < 30 or len(neg) < 30:
        return None, None
    pos = rng.choice(pos, min(len(pos), max_pos), replace=False)
    neg = rng.choice(neg, min(len(neg), len(pos) * neg_ratio), replace=False)
    idx = np.r_[pos, neg]
    return F.reshape(-1, F.shape[-1])[idx], np.r_[np.ones(len(pos)), np.zeros(len(neg))]

def predict_chunks(model, F, mask):
    P = np.full(mask.shape, np.nan, dtype="float32")
    flat = F.reshape(-1, F.shape[-1]); out = P.reshape(-1)
    idx = np.flatnonzero(mask.ravel())
    for ch in np.array_split(idx, max(1, len(idx) // 400000)):
        if len(ch): out[ch] = model.predict_proba(flat[ch])[:, 1]
    return P

def oof_susceptibility(F, y, domain, folds, max_pos, neg_ratio):
    # out-of-fold map: every pixel is predicted by a model that never saw its spatial block
    P = np.full(domain.shape, np.nan, dtype="float32"); models = {}
    for f in np.unique(folds[domain]):
        X, t = sample_xy(F, y, domain, folds, f_excl=f, max_pos=max_pos, neg_ratio=neg_ratio, seed=int(f))
        if X is None:
            continue
        m = make_model().fit(X, t, sample_weight=balanced_w(t)); models[int(f)] = m
        Pf = predict_chunks(m, F, domain & (folds == f))
        P = np.where(np.isfinite(Pf), Pf, P)
    return P, models

def score_metrics(score, y, mask):
    m = mask & np.isfinite(score)
    if m.sum() == 0 or y[m].sum() == 0 or y[m].sum() == m.sum():
        return dict(auc=np.nan, ap=np.nan, base_rate=np.nan, n_pos=0)
    return dict(auc=float(roc_auc_score(y[m], score[m])), ap=float(average_precision_score(y[m], score[m])),
                base_rate=float(y[m].mean()), n_pos=int(y[m].sum()))

def gain_curve(score, y, mask, n=200):
    m = mask & np.isfinite(score)
    s, yy = score[m], y[m].astype(float)
    order = np.argsort(-s); cum = np.cumsum(yy[order]) / max(yy.sum(), 1)
    frac = np.arange(1, len(s) + 1) / len(s)
    idx = np.unique(np.linspace(0, len(s) - 1, n).astype(int))
    return frac[idx], cum[idx]

# ---------- validation maths (stratified sample -> unbiased accuracy with confidence intervals) ----------
def wilson(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan)
    p = k / n; d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d; h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (max(0, c - h), min(1, c + h))

STRATA = ["both", "opt_only", "rad_only", "dry"]
def strata_estimates(df, sizes, n_boot=500, seed=0):
    rng = np.random.default_rng(seed)
    groups = {s: df.loc[df["stratum"] == s, "label"].astype(int).values for s in STRATA}
    def est(g):
        p = {s: (g[s].mean() if len(g[s]) else np.nan) for s in STRATA}
        fl = ["both", "opt_only", "rad_only"]
        TP = sum(sizes[s] * p[s] for s in fl if np.isfinite(p[s])); FP = sum(sizes[s] * (1 - p[s]) for s in fl if np.isfinite(p[s]))
        FN = sizes["dry"] * p["dry"] if np.isfinite(p["dry"]) else np.nan
        def prec(ss):
            num = sum(sizes[s] * p[s] for s in ss if np.isfinite(p[s])); den = sum(sizes[s] for s in ss if np.isfinite(p[s]))
            return num / den if den else np.nan
        pr = TP / (TP + FP) if TP + FP > 0 else np.nan
        rc = TP / (TP + FN) if np.isfinite(FN) and TP + FN > 0 else np.nan
        f1 = 2 * pr * rc / (pr + rc) if np.isfinite(pr) and np.isfinite(rc) and pr + rc > 0 else np.nan
        io = TP / (TP + FP + FN) if np.isfinite(FN) and TP + FP + FN > 0 else np.nan
        return dict(precision=pr, recall=rc, f1=f1, iou=io, precision_optical=prec(["both", "opt_only"]),
                    precision_radar=prec(["both", "rad_only"]), precision_both=prec(["both"]))
    point = est(groups); boots = {k: [] for k in point}
    for _ in range(n_boot):
        g = {s: (rng.choice(v, len(v), replace=True) if len(v) else v) for s, v in groups.items()}
        e = est(g)
        for k in boots: boots[k].append(e[k])
    ci = {}
    for k, v in boots.items():
        v = np.array(v, float)
        ci[k] = (float(np.nanpercentile(v, 2.5)), float(np.nanpercentile(v, 97.5))) if np.isfinite(v).any() else (np.nan, np.nan)
    return point, ci

In [ ]:
import matplotlib, matplotlib.colors as mcolors
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

C = dict(flood_both="#d7191c", flood_one="#fdae61", optical="#2b83ba", radar="#8e44ad", ok="#1a9641", warn="#f4a000", bad="#d7191c")
plt.rcParams.update({"figure.dpi": 110, "axes.titlesize": 12, "axes.titleweight": "bold",
                     "axes.spines.top": False, "axes.spines.right": False, "font.size": 10})

def scalebar(ax, res_m):
    # draws a km scale bar + north arrow on a north-up image axis
    (x0, x1), (yb, yt) = ax.get_xlim(), ax.get_ylim()
    width_m = abs(x1 - x0) * res_m
    length = max([l for l in [250, 500, 1000, 2000, 5000, 10000, 20000, 50000] if l <= 0.3 * width_m] or [250])
    px = length / res_m
    xa, ya = x0 + 0.04 * (x1 - x0), yb - 0.06 * (yb - yt)
    ax.plot([xa, xa + px], [ya, ya], color="white", lw=5, solid_capstyle="butt", zorder=5)
    ax.plot([xa, xa + px], [ya, ya], color="black", lw=2.5, solid_capstyle="butt", zorder=6)
    ax.text(xa + px / 2, ya - 0.015 * (yb - yt), f"{length/1000:g} km", color="white", ha="center", va="bottom",
            fontsize=9, fontweight="bold", bbox=dict(facecolor="black", alpha=0.55, pad=1.5, edgecolor="none"), zorder=7)
    ax.annotate("N", xy=(0.96, 0.95), xytext=(0.96, 0.84), xycoords="axes fraction", textcoords="axes fraction", ha="center",
                color="white", fontweight="bold", arrowprops=dict(arrowstyle="-|>", color="white", lw=2),
                bbox=dict(facecolor="black", alpha=0.55, pad=1.5, edgecolor="none"), zorder=7)

def legend_box(ax, items, loc="lower right", title=None):
    handles = [Patch(facecolor=c, edgecolor="black", label=l) for l, c in items]
    ax.legend(handles=handles, loc=loc, framealpha=0.95, fontsize=9, title=title)

def callout(ax, text, color="#fff8dc"):
    ax.text(0.5, -0.12, text, transform=ax.transAxes, ha="center", va="top", fontsize=9, style="italic",
            bbox=dict(facecolor=color, edgecolor="#cccccc", boxstyle="round,pad=0.4"), wrap=True)

## 2. Feasibility check: is there usable satellite data for each event?
A flood event with no good before/after image is a dead end, so we check first.

Things to know:
- Only one radar satellite flew between early 2022 and Dec 2024 (12-day revisit), so **2024 is the weakest year for radar**. Two satellites (6-day) cover 2016-2021 and 2025 onward.
- Sentinel-2A alone flew in 2016 (about 10-day revisit).
- `bbox` = [west, south, east, north] in degrees. These are **approximate**; confirm in Section 4.

In [ ]:
EVENTS = {
    # key: (label, event date, bbox) -> dates from news reports / Wikipedia; boxes are approximate
    "hurghada_2016":  dict(label="Hurghada, Red Sea flash flood",           date="2016-10-27", bbox=[33.60, 27.10, 34.00, 27.45]),
    "ras_gharib_2016":dict(label="Ras Gharib, Red Sea flash flood",         date="2016-10-27", bbox=[32.95, 28.20, 33.35, 28.50]),
    "sinai_2020":     dict(label="South Sinai / Sharm storm (location unverified)", date="2020-03-12", bbox=[34.10, 27.70, 34.50, 28.00]),
    "aswan_2024":     dict(label="Aswan eastern wadis (location approximate)",       date="2024-08-09", bbox=[32.90, 23.90, 33.30, 24.20]),
    "alex_2025":      dict(label="Alexandria hail + flash flood (urban: expect weak radar)", date="2025-05-31", bbox=[29.80, 31.10, 30.10, 31.30]),
}
PRE_DAYS, POST_DAYS = 60, 12     # before-window length, after-window length (days)

def windows(date):
    d = pd.Timestamp(date)
    f = lambda t: t.strftime("%Y-%m-%d")
    return (f(d - pd.Timedelta(days=PRE_DAYS)), f(d - pd.Timedelta(days=1))), (f(d), f(d + pd.Timedelta(days=POST_DAYS)))

def search_items(collection, bbox, rng, **kw):
    try:
        s = catalog.search(collections=[collection], bbox=bbox, datetime=f"{rng[0]}/{rng[1]}", **kw)
        return list(s.item_collection())
    except Exception as e:
        print(f"   ! search failed for {collection}: {e}")
        return []

In [ ]:
# Optional: re-run the feasibility table (takes about a minute). Skip if you already know your events.
rows = []
for key, ev in EVENTS.items():
    pre, post = windows(ev["date"])
    s2_pre  = search_items("sentinel-2-l2a", ev["bbox"], pre,  query={"eo:cloud_cover": {"lt": 40}})
    s2_post = search_items("sentinel-2-l2a", ev["bbox"], post, query={"eo:cloud_cover": {"lt": 40}})
    s1_pre  = search_items("sentinel-1-rtc", ev["bbox"], pre)
    s1_post = search_items("sentinel-1-rtc", ev["bbox"], post)
    first = lambda items: min((i.datetime.date() for i in items), default=None)
    rows.append(dict(event=key, date=ev["date"], S2_before=len(s2_pre), S2_after=len(s2_post), first_S2_after=first(s2_post),
                     S1_before=len(s1_pre), S1_after=len(s1_post), first_S1_after=first(s1_post)))
pd.DataFrame(rows)

## 3. Choose the events and tune the knobs
- **PRIMARY** is the case study (full analysis). **SECONDARY** is a *different place hit by the same storm*, used to test whether the method and the ML generalise **with thresholds unchanged**.
- Thresholds below match your report (radar drop of 5 dB). Change them here only.

In [ ]:
PRIMARY, SECONDARY = "ras_gharib_2016", "sinai_2020"
RES = 20                         # metres per pixel (20 keeps it fast; 10 is sharper but 4x slower)
MAX_SCENES = 6                   # max image dates used per window

OPT = dict(d_thr=0.15, post_thr=0.0, pre_max=0.0)         # optical rule (MNDWI change)
POST_AGG = "max"                                           # "max" = water seen in ANY clear after-image; "median" = more conservative
SAR = dict(post_thr=-18.0, diff_thr=-5.0, pre_min=-17.0)  # radar rule (VV, dB)
MAX_SLOPE_DEG, MAX_REL_ELEV_M = 5.0, 15.0                  # terrain exclusion
SEA_BUFFER_M = 100                                         # exclude sea + this much coast (sediment plumes, shoreline)
FUSION = "union"                                           # "union" = either sensor; "intersection" = both
MIN_PIXELS = 10                                            # remove blobs smaller than this

POST_DAYS_EXT = 14               # window for the temporal test (Section 9d)
TEMPORAL_MAX_DATES = 6
ML = dict(block_m=2000, folds=5, max_pos=20000, neg_ratio=4, drain_km2=0.25)
print("PRIMARY:", PRIMARY, "|", EVENTS[PRIMARY]["label"]); print("SECONDARY:", SECONDARY, "|", EVENTS[SECONDARY]["label"])

## 4. Data access and the detection pipeline (all in functions, so we can run it on any event)
`process_event(key)` does everything for one storm: finds and loads Sentinel-2 and Sentinel-1, loads the terrain and land cover, builds the sea and terrain masks, runs both detectors, fuses them, and records an **audit** of how each step changed the area.

In [ ]:
def make_ctx(key):
    ev = EVENTS[key]
    pre_rng, post_rng = windows(ev["date"])
    return dict(key=key, label=ev["label"], date=ev["date"], bbox=ev["bbox"], epsg=utm_epsg(ev["bbox"]), res=RES, pre_rng=pre_rng, post_rng=post_rng)

def _load(items, bands, ctx, resampling="nearest"):
    return odc_load(items, bands=bands, bbox=ctx["bbox"], crs=f"EPSG:{ctx['epsg']}", resolution=ctx["res"], groupby="solar_day", resampling=resampling)

# ---------------- optical ----------------
def group_by_date(items):
    g = {}
    for it in items:
        g.setdefault(it.datetime.strftime("%Y-%m-%d"), []).append(it)
    return g

def load_s2_date(items, ctx):
    ds = _load(items, ["B02", "B03", "B04", "B08", "B11", "SCL"], ctx).isel(time=0).compute()
    bl = str(items[0].properties.get("s2:processing_baseline", "00.00"))
    off = 1000 if bl >= "04.00" else 0
    R_ = {b: reflectance(ds[b].values, off) for b in ["B02", "B03", "B04", "B08", "B11"]}
    scl = np.nan_to_num(ds["SCL"].values.astype("float32"), nan=0).astype(int)
    clear = np.isin(scl, [2, 4, 5, 6, 7])
    mndwi = norm_diff(R_["B03"], R_["B11"]); mndwi[~clear] = np.nan
    ndvi = norm_diff(R_["B08"], R_["B04"]);  ndvi[~clear] = np.nan
    return dict(mndwi=mndwi, ndvi=ndvi, rgb=np.dstack([R_["B04"], R_["B03"], R_["B02"]]), clear_frac=float(clear.mean()),
                xs=ds.x.values, ys=ds.y.values)

def load_window(items, ctx, tag, verbose=True):
    groups = group_by_date(items)
    ev = pd.Timestamp(ctx["date"])
    dates = sorted(sorted(groups), key=lambda d: abs((pd.Timestamp(d) - ev).days))[:MAX_SCENES]
    out = {}
    for d in sorted(dates):
        try:
            out[d] = load_s2_date(groups[d], ctx)
            if verbose: print(f"  {tag} {d}: clear pixels {out[d]['clear_frac']*100:.0f}%")
        except Exception as e:
            print(f"  ! {tag} {d} failed: {e}")
    return out

def optical_composites(ctx, verbose=True):
    q = {"eo:cloud_cover": {"lt": 40}}
    pre_items  = search_items("sentinel-2-l2a", ctx["bbox"], ctx["pre_rng"],  query=q)
    post_items = search_items("sentinel-2-l2a", ctx["bbox"], ctx["post_rng"], query=q)
    if verbose: print(" Sentinel-2 BEFORE scenes...");
    S2_pre = load_window(pre_items, ctx, "before", verbose)
    if verbose: print(" Sentinel-2 AFTER scenes...")
    S2_post = load_window(post_items, ctx, "after", verbose)
    if not S2_pre or not S2_post:
        return None
    stack = lambda D, k: np.stack([v[k] for v in D.values()])
    agg = np.nanmax if POST_AGG == "max" else np.nanmedian
    a = next(iter(S2_pre.values()))
    return dict(S2_pre=S2_pre, S2_post=S2_post, xs=a["xs"], ys=a["ys"],
                mndwi_pre=np.nanmedian(stack(S2_pre, "mndwi"), axis=0), mndwi_post=agg(stack(S2_post, "mndwi"), axis=0),
                ndvi_pre=np.nanmedian(stack(S2_pre, "ndvi"), axis=0), ndvi_post=np.nanmedian(stack(S2_post, "ndvi"), axis=0),
                best_pre=max(S2_pre.values(), key=lambda v: v["clear_frac"]), best_post=max(S2_post.values(), key=lambda v: v["clear_frac"]))

# ---------------- radar ----------------
def s1_candidates(ctx, rng):
    return search_items("sentinel-1-rtc", ctx["bbox"], rng)

def vv_db_stack(items, ctx):
    ds = _load(items, ["vv", "vh"], ctx, "bilinear").compute()
    return [despeckle_db(to_db(ds.vv.isel(time=t).values)) for t in range(ds.sizes["time"])], ds.x.values, ds.y.values

def same_orbit(items, ro, st):
    return [i for i in items if i.properties.get("sat:relative_orbit") == ro and i.properties.get("sat:orbit_state") == st]

def radar_pair(ctx, verbose=True):
    pre_all, post_all = s1_candidates(ctx, ctx["pre_rng"]), s1_candidates(ctx, ctx["post_rng"])
    if verbose: print(f" radar items found: {len(pre_all)} before, {len(post_all)} after")
    for post in sorted(post_all, key=lambda i: i.datetime):
        ro, st = post.properties.get("sat:relative_orbit"), post.properties.get("sat:orbit_state")
        pre_m = same_orbit(pre_all, ro, st)
        if not pre_m:
            continue
        day = post.datetime.strftime("%Y-%m-%d")
        post_items = [i for i in same_orbit(post_all, ro, st) if i.datetime.strftime("%Y-%m-%d") == day]
        pre_days_ = sorted({i.datetime.strftime("%Y-%m-%d") for i in pre_m})[-3:]
        pre_items = [i for i in pre_m if i.datetime.strftime("%Y-%m-%d") in pre_days_]
        if verbose: print(f" Using orbit {ro} ({st}); AFTER {day}; BEFORE {pre_days_}")
        try:
            pre_stack, xs_, ys_ = vv_db_stack(pre_items, ctx); post_stack, _, _ = vv_db_stack(post_items, ctx)
            return dict(vv_pre_db=np.nanmedian(np.stack(pre_stack), axis=0), vv_post_db=post_stack[0], post_day=day, pre_days=pre_days_,
                        orbit=f"{ro} {st}", xs=xs_, ys=ys_)
        except Exception as e:
            print(" Radar load failed:", e); return None
    if verbose: print(" No same-orbit before/after radar pair found.")
    return None

# ---------------- terrain + land cover ----------------
def load_dem(ctx):
    items = search_items("cop-dem-glo-30", ctx["bbox"], ("1900-01-01", "2100-01-01"))
    return _load(items, ["data"], ctx, "bilinear").isel(time=0).compute()["data"].values.astype("float32")

def load_landcover(ctx):
    items = sorted(search_items("esa-worldcover", ctx["bbox"], ("2020-01-01", "2022-12-31")), key=lambda i: i.datetime)[-1:]
    return _load(items, ["map"], ctx).isel(time=0).compute()["map"].values

def make_sea_mask(lc, mndwi_pre, res, buffer_m, shape):
    sea = np.zeros(shape, bool)
    if lc is not None:
        w = (lc == 80)
        if buffer_m > 0 and w.any():
            w = ndi.binary_dilation(w, iterations=max(1, int(round(buffer_m / res))))
        sea |= w
    if mndwi_pre is not None:
        sea |= np.nan_to_num(mndwi_pre, nan=-1) > 0.2          # water that was already there before the storm
    return sea

# ---------------- detection + fusion ----------------
def fuse_masks(A, opt_p, sar_p, fusion=None, min_pixels=None):
    fusion = fusion or FUSION; min_pixels = MIN_PIXELS if min_pixels is None else min_pixels
    shape = A["shape"]; Z = np.zeros(shape, bool)
    fo = optical_flood(A["mndwi_pre"], A["mndwi_post"], **opt_p) if A["have_opt"] else Z.copy()
    fs = radar_flood(A["vv_pre_db"], A["vv_post_db"], **sar_p) if A["have_sar"] else Z.copy()
    out = dict(opt_raw=fo.copy(), sar_raw=fs.copy())
    fo &= A["topo_ok"]; fs &= A["topo_ok"]
    out.update(opt_terrain=fo.copy(), sar_terrain=fs.copy())
    fo &= ~A["sea"]; fs &= ~A["sea"]
    if A["have_opt"] and A["have_sar"]:
        fused = (fo | fs) if fusion == "union" else (fo & fs)
    else:
        fused = fo if A["have_opt"] else fs
    out.update(opt=fo, sar=fs, fused_pre_clean=fused, flood=clean_mask(fused, min_pixels))
    return out

def process_event(key, verbose=True):
    ctx = make_ctx(key)
    if verbose: print(f"=== {key}: {ctx['label']} | storm {ctx['date']} | UTM {ctx['epsg']} ===")
    opt = optical_composites(ctx, verbose)
    rad = radar_pair(ctx, verbose)
    if opt is None and rad is None:
        raise RuntimeError(f"No usable satellite data for {key}. Check the feasibility table or move the bbox.")
    src = opt if opt is not None else rad
    xs_, ys_ = src["xs"], src["ys"]; shape = (len(ys_), len(xs_))
    A = dict(ctx=ctx, xs=xs_, ys=ys_, shape=shape, have_opt=opt is not None, have_sar=rad is not None)
    if opt is not None:
        A.update({k: opt[k] for k in ["S2_pre", "S2_post", "mndwi_pre", "mndwi_post", "ndvi_pre", "ndvi_post", "best_pre", "best_post"]})
    else:
        A.update(S2_pre={}, S2_post={}, mndwi_pre=None, mndwi_post=None, ndvi_pre=None, ndvi_post=None, best_pre=None, best_post=None)
    if rad is not None:
        A.update(vv_pre_db=fit_shape(rad["vv_pre_db"], shape), vv_post_db=fit_shape(rad["vv_post_db"], shape),
                 post_day=rad["post_day"], pre_days=rad["pre_days"], orbit=rad["orbit"])
    else:
        A.update(vv_pre_db=None, vv_post_db=None, post_day=None, pre_days=[], orbit=None)
    for k in ("mndwi_pre", "mndwi_post", "ndvi_pre", "ndvi_post"):
        if A[k] is not None: A[k] = fit_shape(A[k], shape)
    # terrain
    A["dem"], A["have_dem"] = None, False
    try:
        A["dem"] = fit_shape(load_dem(ctx), shape); A["have_dem"] = True
        A["slope"] = slope_deg(A["dem"], RES); A["rel"] = rel_elevation(A["dem"], RES)
        A["topo_ok"] = (A["slope"] < MAX_SLOPE_DEG) & (A["rel"] < MAX_REL_ELEV_M)
    except Exception as e:
        print(" DEM unavailable -> no terrain mask. Reason:", e)
        A["slope"] = A["rel"] = None; A["topo_ok"] = np.ones(shape, bool)
    # land cover + sea
    try:
        A["lc"] = fit_shape(load_landcover(ctx), shape)
    except Exception as e:
        print(" Land cover unavailable -> sea mask uses pre-storm water only. Reason:", e); A["lc"] = None
    A["sea"] = make_sea_mask(A["lc"], A["mndwi_pre"], RES, SEA_BUFFER_M, shape)
    # masks
    M = fuse_masks(A, OPT, SAR); A.update(M)
    px = (RES / 1000.0) ** 2; A["px_km2"] = px
    both = M["opt"] & M["sar"]
    A["audit"] = dict(
        optical_raw_km2=M["opt_raw"].sum() * px, radar_raw_km2=M["sar_raw"].sum() * px,
        optical_after_terrain_km2=M["opt_terrain"].sum() * px, radar_after_terrain_km2=M["sar_terrain"].sum() * px,
        sea_removed_km2=((M["opt_terrain"] | M["sar_terrain"]) & A["sea"]).sum() * px,
        optical_final_km2=M["opt"].sum() * px, radar_final_km2=M["sar"].sum() * px, overlap_km2=both.sum() * px,
        union_before_cleaning_km2=(M["opt"] | M["sar"]).sum() * px, fused_before_cleaning_km2=M["fused_pre_clean"].sum() * px,
        fused_final_km2=M["flood"].sum() * px)
    A["audit"]["removed_small_blobs_km2"] = A["audit"]["fused_before_cleaning_km2"] - A["audit"]["fused_final_km2"]
    A["audit"] = {k: round(float(v), 3) for k, v in A["audit"].items()}
    A["confidence"] = M["opt"].astype(int) + M["sar"].astype(int)
    A["summary"] = {
        "event": key, "event_date": ctx["date"], "bbox": ctx["bbox"], "resolution_m": RES,
        "sensors_used": [s for s, ok in [("Sentinel-2", A["have_opt"]), ("Sentinel-1", A["have_sar"])] if ok],
        "terrain_mask": A["have_dem"], "sea_mask": True,
        "flooded_km2_fused": A["audit"]["fused_final_km2"], "flooded_km2_optical": A["audit"]["optical_final_km2"],
        "flooded_km2_radar": A["audit"]["radar_final_km2"], "both_sensors_agree_km2": round(float((both & M["flood"]).sum() * px), 3),
        "audit": A["audit"]}
    if verbose: print(f" -> fused flooded area: {A['audit']['fused_final_km2']} km²  (sensors: {', '.join(A['summary']['sensors_used'])})")
    return A

def activate(R):
    # expose one event's arrays as notebook-level variables (used by the figure cells below)
    g = globals()
    c = R["ctx"]
    g.update(ACTIVE=c["key"], EV=dict(label=c["label"], date=c["date"], bbox=c["bbox"]), BBOX=c["bbox"], EVENT_DATE=c["date"], EPSG=c["epsg"],
             xs=R["xs"], ys=R["ys"], shape=R["shape"], px_km2=R["px_km2"], HAVE_OPTICAL=R["have_opt"], HAVE_RADAR=R["have_sar"], HAVE_DEM=R["have_dem"],
             S2_pre=R["S2_pre"], S2_post=R["S2_post"], best_pre=R["best_pre"], best_post=R["best_post"],
             mndwi_pre=R["mndwi_pre"], mndwi_post=R["mndwi_post"], ndvi_pre=R["ndvi_pre"], ndvi_post=R["ndvi_post"],
             vv_pre_db=R["vv_pre_db"], vv_post_db=R["vv_post_db"], pre_days=R["pre_days"], post_day=R["post_day"],
             s1_after=[R["post_day"]] if R["post_day"] else [], dem=R["dem"], slope=R["slope"], rel=R["rel"], topo_ok=R["topo_ok"],
             sea_mask=R["sea"], lc=R["lc"], flood_opt_raw=R["opt_raw"], flood_sar_raw=R["sar_raw"], flood_opt=R["opt"], flood_sar=R["sar"],
             flood=R["flood"], confidence=R["confidence"], summary=R["summary"], audit=R["audit"])

## 5. Run the primary event
This is the slow cell (downloads). Everything after it is analysis on data already in memory.

In [ ]:
R1 = process_event(PRIMARY)
activate(R1)

### 5a. Sanity check: BEFORE / AFTER and what changed

In [ ]:
if HAVE_OPTICAL:
    fig, ax = plt.subplots(1, 3, figsize=(18, 6))
    ax[0].imshow(np.clip(best_pre["rgb"] * 3.5, 0, 1));  ax[0].set_title("BEFORE (true colour)")
    ax[1].imshow(np.clip(best_post["rgb"] * 3.5, 0, 1)); ax[1].set_title("AFTER (true colour)")
    im = ax[2].imshow(mndwi_post - mndwi_pre, cmap="RdBu", vmin=-0.5, vmax=0.5); ax[2].set_title("Change in water index (blue = more water)")
    plt.colorbar(im, ax=ax[2], fraction=0.04)
    for a in ax: a.axis("off"); scalebar(a, RES)
    plt.suptitle(f"{EV['label']}: does this box show the area you expect?", fontsize=13)
    plt.tight_layout(); plt.show()
else:
    print("No optical data for this event; skipping.")

In [ ]:
if HAVE_OPTICAL:
    d = (mndwi_post - mndwi_pre)
    fig, ax = plt.subplots(1, 3, figsize=(20, 5.8), gridspec_kw={"width_ratios": [1, 1, 1.1]})
    for a, arr, t in [(ax[0], mndwi_pre, "BEFORE: wetness (MNDWI)"), (ax[1], mndwi_post, "AFTER: wetness (MNDWI)")]:
        im = a.imshow(arr, cmap="BrBG", vmin=-0.6, vmax=0.6); a.set_title(t); a.axis("off")
        scalebar(a, RES)
    cb = plt.colorbar(im, ax=ax[:2], fraction=0.025, pad=0.02); cb.set_label("brown = dry   |   green-blue = wet / water")
    dd = d[np.isfinite(d)]
    ax[2].hist(dd, bins=120, range=(-1, 1), color="#bbbbbb")
    ax[2].axvline(OPT["d_thr"], color=C["bad"], lw=2, ls="--")
    ax[2].axvspan(OPT["d_thr"], 1, color=C["optical"], alpha=0.25)
    ax[2].set_yscale("log"); ax[2].set_xlabel("change in wetness (AFTER minus BEFORE)"); ax[2].set_ylabel("number of pixels (log scale)")
    ax[2].set_title("Which pixels got wetter?")
    ax[2].text(0.02, 0.95, "tall bar near 0 = most land did not change", transform=ax[2].transAxes, va="top", fontsize=9)
    ax[2].text(OPT["d_thr"] + 0.02, ax[2].get_ylim()[1] * 0.3, f"counted as new water\n(change > {OPT['d_thr']})", color=C["optical"], fontsize=9, va="top")
    plt.show()

> 👀 **What to look for:** brown = dry, green/blue = wet. The AFTER map should show *new* wet streaks (valley floors, wadi channels) that were brown BEFORE. In the histogram the shaded blue zone on the right is what the notebook calls "new water". If it contains almost nothing, the event/box may be wrong; if it contains a huge chunk of the picture, the threshold is too loose.

### 5b. Radar view and satellite timing

In [ ]:
if HAVE_RADAR:
    fig, ax = plt.subplots(1, 2, figsize=(14, 4.2))
    for arr, lab, col in [(vv_pre_db, "BEFORE", "#666666"), (vv_post_db, "AFTER", C["optical"])]:
        v_ = arr[np.isfinite(arr)]
        ax[0].hist(v_, bins=100, range=(-35, 5), alpha=0.55, color=col, label=lab)
    ax[0].axvline(SAR["post_thr"], color=C["bad"], ls="--", lw=2)
    ax[0].text(SAR["post_thr"] - 0.5, ax[0].get_ylim()[1] * 0.9, "water-dark zone", ha="right", color=C["bad"], fontsize=9)
    ax[0].set_xlabel("radar brightness (dB): left = darker"); ax[0].set_ylabel("pixels"); ax[0].legend()
    ax[0].set_title("Radar brightness before vs after")
    dv = (vv_post_db - vv_pre_db); dv = dv[np.isfinite(dv)]
    ax[1].hist(dv, bins=100, range=(-12, 12), color="#bbbbbb")
    ax[1].axvline(SAR["diff_thr"], color=C["bad"], ls="--", lw=2)
    ax[1].axvspan(-12, SAR["diff_thr"], color=C["optical"], alpha=0.25)
    ax[1].set_xlabel("change in brightness (dB); negative = got darker"); ax[1].set_title("Which pixels got darker?")
    plt.tight_layout(); plt.show()

> 👀 **What to look for:** in the left plot the AFTER curve should grow a bump on the dark (left) side compared with BEFORE: that bump is new water. *Careful:* dry sand also sits in the dark zone, which is why we require a **drop** (right plot, blue zone) and not just darkness.

In [ ]:
# Timeline: when did the satellites actually look, relative to the storm?
ev = pd.Timestamp(EVENT_DATE)
s1_before = pre_days if HAVE_RADAR else []
s1_after  = [post_day] if HAVE_RADAR else []
fig, ax = plt.subplots(figsize=(12, 3.2))
ax.axvspan(ev, ev + pd.Timedelta(days=POST_DAYS), color="#fee8c8", alpha=0.7)
ax.axvline(ev, color="black", lw=3); ax.text(ev, 1.32, "STORM", ha="center", fontweight="bold")
def dots(dates, y, color, marker, label):
    if dates:
        ax.scatter([pd.Timestamp(d) for d in dates], [y] * len(dates), s=110, color=color, marker=marker, label=label, zorder=5, edgecolor="black")
dots(list(S2_pre.keys()),  0.35, C["optical"], "o", "Sentinel-2 BEFORE")
dots(list(S2_post.keys()), 0.35, C["optical"], "s", "Sentinel-2 AFTER")
dots(s1_before, 0.95, C["radar"], "o", "Sentinel-1 BEFORE")
dots(s1_after,  0.95, C["radar"], "s", "Sentinel-1 AFTER")
ax.set_yticks([0.35, 0.95]); ax.set_yticklabels(["optical (camera)", "radar"]); ax.set_ylim(0, 1.5)
import matplotlib.dates as mdates
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b")); ax.xaxis.set_major_locator(mdates.AutoDateLocator(maxticks=8))
ax.legend(ncol=4, loc="lower center", bbox_to_anchor=(0.5, -0.5), fontsize=9, frameon=False)
firsts = []
for lab, ds_ in [("optical", list(S2_post.keys())), ("radar", s1_after)]:
    if ds_: firsts.append(f"first {lab} AFTER image: +{(pd.Timestamp(min(ds_)) - ev).days} days")
ax.set_title("When did the satellites look?   " + "   |   ".join(firsts) if firsts else "When did the satellites look?")
plt.tight_layout(); plt.show()

> 👀 **What to look for:** circles = before the storm, squares = after. The shaded band is the after-window. The closer the first square sits to the black STORM line, the more likely the water was still there when the satellite passed. A gap of more than about a week means flash-flood water may already have drained.

### 5c. Terrain and sea masks: remove the impossible
Two exclusions are applied **before** counting any flooded area:
1. **Terrain:** water cannot pool on steep or high ground.
2. **Sea and coast** (new): the Red Sea and its sediment plumes are not "flooded land". We exclude ESA WorldCover water plus a small coastal buffer, and anything that was already water before the storm.

In [ ]:
if HAVE_DEM:
    fig, ax = plt.subplots(1, 4, figsize=(22, 5))
    for a, arr, t, cm in zip(ax, [dem, slope, topo_ok, sea_mask], ["Elevation (m)", "Slope (deg)", "Flood-possible terrain", "Sea / coast mask (excluded)"],
                             ["terrain", "magma", "gray", "Blues"]):
        im = a.imshow(arr, cmap=cm); a.set_title(t); a.axis("off"); plt.colorbar(im, ax=a, fraction=0.04)
    plt.tight_layout(); plt.show()
    print(f"Terrain keeps {topo_ok.mean()*100:.0f}% of the box; the sea/coast mask covers {sea_mask.mean()*100:.0f}%.")
else:
    print("No terrain data for this event.")

## 6. Numbers audit: how the raw detections become the final area
Every area in your report should be traceable to one bar below. This answers "why is the fused total smaller than optical + radar?"

In [ ]:
a = audit
steps = [
    ("Optical raw",              a["optical_raw_km2"],         C["optical"]),
    ("Radar raw",                a["radar_raw_km2"],           C["radar"]),
    ("Optical after terrain",    a["optical_after_terrain_km2"], C["optical"]),
    ("Radar after terrain",      a["radar_after_terrain_km2"],   C["radar"]),
    ("Optical after sea mask",   a["optical_final_km2"],       C["optical"]),
    ("Radar after sea mask",     a["radar_final_km2"],         C["radar"]),
    ("Union (either sensor)",    a["union_before_cleaning_km2"], "#666666"),
    ("FINAL after speck removal", a["fused_final_km2"],        C["flood_both"]),
]
fig, ax = plt.subplots(1, 2, figsize=(17, 5.2), gridspec_kw={"width_ratios": [1.5, 1]})
ax[0].barh([s[0] for s in steps][::-1], [s[1] for s in steps][::-1], color=[s[2] for s in steps][::-1])
for i_, s in enumerate(steps[::-1]): ax[0].text(s[1], i_, f"  {s[1]:.2f} km²", va="center", fontsize=10)
ax[0].set_xlabel("km²"); ax[0].set_title("From raw detections to the final flood area"); ax[0].set_xlim(0, max(s[1] for s in steps) * 1.25 + 1e-9)
removed = [("Terrain\n(steep/high ground)", (a["optical_raw_km2"] + a["radar_raw_km2"]) - (a["optical_after_terrain_km2"] + a["radar_after_terrain_km2"])),
           ("Sea / coast", (a["optical_after_terrain_km2"] + a["radar_after_terrain_km2"]) - (a["optical_final_km2"] + a["radar_final_km2"])),
           ("Tiny specks\n(< %d px)" % MIN_PIXELS, a["removed_small_blobs_km2"])]
ax[1].bar([r[0] for r in removed], [max(r[1], 0) for r in removed], color=["#bbbbbb", C["optical"], "#ddaa33"])
for i_, r in enumerate(removed): ax[1].text(i_, max(r[1], 0), f"{max(r[1], 0):.2f}", ha="center", va="bottom")
ax[1].set_ylabel("km² removed (sensor areas added together)"); ax[1].set_title("What each clean-up step removed")
plt.tight_layout(); plt.show()
print(f"Overlap of the two sensors: {a['overlap_km2']} km²  |  union before cleaning: {a['union_before_cleaning_km2']} km²  |  FINAL: {a['fused_final_km2']} km²")
spk = a["removed_small_blobs_km2"] / a["fused_before_cleaning_km2"] if a["fused_before_cleaning_km2"] else 0
print(f"Speck removal deleted {spk*100:.0f}% of the pre-cleaning area.", "That is high: detections are fragmented, so expect noise." if spk > 0.2 else "Detections are mostly coherent blobs.")

> 👀 **What to look for:** the **sea mask** bar tells you how much of the first version's area was ocean sediment (this should now be zero in the final number). A large **tiny specks** bar means a lot of the raw detections were isolated pixels, i.e. noise rather than channels. Quote the **FINAL** bar in your report, and mention which clean-ups were applied.

## 7. The flood map and who detected what

In [ ]:
def gray3(a):
    g = np.clip((np.nan_to_num(a, nan=-25) + 25) / 25, 0, 1)
    return np.dstack([g, g, g])
if HAVE_OPTICAL:
    bg_pre, bg_post = np.clip(best_pre["rgb"] * 3.5, 0, 1), np.clip(best_post["rgb"] * 3.5, 0, 1)
else:
    bg_pre, bg_post = gray3(vv_pre_db), gray3(vv_post_db)
overlay = bg_post.copy()
overlay[flood & (confidence <= 1)] = mcolors.to_rgb(C["flood_one"])
overlay[flood & (confidence == 2)] = mcolors.to_rgb(C["flood_both"])
fig, ax = plt.subplots(1, 2, figsize=(17, 7.5))
ax[0].imshow(bg_post); ax[0].set_title("After the storm (what the satellite saw)")
ax[1].imshow(overlay); ax[1].set_title(f"Detected flood: {summary['flooded_km2_fused']} km²")
legend_box(ax[1], [("Both satellites agree", C["flood_both"]), ("One satellite only", C["flood_one"])], title="Confidence")
for a_ in ax: a_.axis("off"); scalebar(a_, RES)
plt.tight_layout(); plt.savefig("flood_map.png", dpi=150); plt.show()
print(f"In everyday terms: about {summary['flooded_km2_fused']} km², roughly {summary['flooded_km2_fused']*1e6/7140:,.0f} football pitches of newly flooded ground.")

> 👀 **What to look for (flood map):** real flash floods trace valleys and low ground in branching, river-like shapes. Large, perfectly round or rectangular blobs, or red on hillsides, are warning signs. Red (both satellites) should sit inside orange regions.

### 7b. Who found what, and what did the terrain mask remove?

In [ ]:
cls = np.zeros(shape, int)
cls[flood & flood_opt & ~flood_sar] = 1
cls[flood & ~flood_opt & flood_sar] = 2
cls[flood & flood_opt & flood_sar] = 3
CLS_COL  = {1: C["optical"], 2: C["radar"], 3: C["flood_both"]}
CLS_NAME = {1: "Optical only", 2: "Radar only", 3: "Both agree"}
rgba = np.zeros(shape + (4,))
for k, col in CLS_COL.items():
    rgba[cls == k] = mcolors.to_rgba(col, 1.0)

fig, ax = plt.subplots(1, 3, figsize=(21, 6.2), gridspec_kw={"width_ratios": [1.5, 1, 1]})
ax[0].imshow(bg_post); ax[0].imshow(rgba); ax[0].axis("off"); scalebar(ax[0], RES)
ax[0].set_title("Who detected each flooded pixel?")
legend_box(ax[0], [(CLS_NAME[k], CLS_COL[k]) for k in (1, 2, 3)])
areas = {CLS_NAME[k]: float((cls == k).sum() * px_km2) for k in (1, 2, 3)}
bars = ax[1].barh(list(areas.keys()), list(areas.values()), color=[CLS_COL[k] for k in (1, 2, 3)])
for b_, v_ in zip(bars, areas.values()): ax[1].text(v_, b_.get_y() + b_.get_height() / 2, f" {v_:.2f} km²", va="center")
ax[1].set_xlabel("flooded area (km²)"); ax[1].set_title("Area by agreement"); ax[1].set_xlim(0, max(areas.values() or [1]) * 1.3 + 1e-9)
if HAVE_DEM:
    lab_ = ["Optical", "Radar"]
    raw_ = [flood_opt_raw.sum() * px_km2, flood_sar_raw.sum() * px_km2]; kept_ = [flood_opt.sum() * px_km2, flood_sar.sum() * px_km2]
    xx = np.arange(2); w_ = 0.38
    ax[2].bar(xx - w_ / 2, raw_, w_, color="#cccccc", label="before terrain mask"); ax[2].bar(xx + w_ / 2, kept_, w_, color=[C["optical"], C["radar"]], label="after terrain mask")
    for i_, (r_, k_) in enumerate(zip(raw_, kept_)):
        ax[2].text(i_, max(r_, k_) * 1.02, f"removed {max(r_ - k_, 0):.2f} km²\n(impossible)", ha="center", va="bottom", fontsize=9, color=C["bad"])
    ax[2].set_ylim(0, max(raw_ + kept_ + [1e-6]) * 1.4)
    ax[2].set_xticks(xx); ax[2].set_xticklabels(lab_); ax[2].set_ylabel("km²"); ax[2].legend(fontsize=9); ax[2].set_title("What the terrain mask removed")
else:
    ax[2].axis("off"); ax[2].text(0.5, 0.5, "No terrain data loaded", ha="center")
plt.tight_layout(); plt.show()

> 👀 **What to look for:** *Left:* red is the most trustworthy core. *Middle:* if "Both agree" is small, the sensors see different things (the temporal test in Section 9d examines whether that is timing or error). *Right:* the grey bar minus the coloured bar is false alarms the terrain rule deleted.

### 7c. Zoom in on the biggest flooded areas (BEFORE / AFTER / detection)

In [ ]:
blobs = top_components(flood, n=3)
if not blobs:
    print("No flooded areas large enough to zoom into.")
else:
    fig, axes = plt.subplots(len(blobs), 3, figsize=(16, 5.2 * len(blobs)), squeeze=False)
    for i, (px_, (r0, r1, c0, c1)) in enumerate(blobs):
        m_ = flood[r0:r1, c0:c1]
        for j, (img, t) in enumerate([(bg_pre, "BEFORE"), (bg_post, "AFTER"), (bg_post, "AFTER + detected flood outline")]):
            a = axes[i, j]; a.imshow(img[r0:r1, c0:c1]); a.axis("off")
            a.set_title(f"Zoom #{i+1} ({px_*px_km2:.2f} km²): {t}", fontsize=11)
        axes[i, 2].contour(m_.astype(float), levels=[0.5], colors=[C["flood_both"]], linewidths=2)
        scalebar(axes[i, 1], RES)
    plt.tight_layout(); plt.show()

> 👀 **What to look for:** the zoom-ins are your best *eyeball validation*. Between BEFORE and AFTER you should see a visible change **inside** the red outline (darker/wetter ground, ponded water, a coloured channel). If BEFORE and AFTER look the same inside the outline, that region is a false alarm. Screenshot good and bad examples: they make strong slides.

### 7d. How sensitive is the answer to my thresholds? (curves)

In [ ]:
topo = (topo_ok if HAVE_DEM else np.ones(shape, bool)) & ~sea_mask
def area_curve(fn, values):
    return [float((fn(v) & topo).sum() * px_km2) for v in values]

fig, ax = plt.subplots(1, 2, figsize=(14, 4.2))
if HAVE_OPTICAL:
    vals = np.linspace(0.05, 0.50, 19)
    cur = area_curve(lambda v: optical_flood(mndwi_pre, mndwi_post, **{**OPT, "d_thr": v}), vals)
    ax[0].plot(vals, cur, "-o", color=C["optical"]); ax[0].axvline(OPT["d_thr"], color=C["bad"], ls="--")
    ax[0].axvspan(OPT["d_thr"] * 0.8, OPT["d_thr"] * 1.2, color=C["bad"], alpha=0.12, label="±20% around my choice")
    ax[0].set_xlabel("optical threshold (change in water index)"); ax[0].set_ylabel("flooded area (km²)"); ax[0].legend(); ax[0].set_title("Optical: area vs threshold")
else:
    ax[0].axis("off")
if HAVE_RADAR:
    vals_r = np.linspace(-1.0, -8.0, 15)
    cur_r = area_curve(lambda v: radar_flood(vv_pre_db, vv_post_db, **{**SAR, "diff_thr": v}), vals_r)
    ax[1].plot(vals_r, cur_r, "-o", color=C["radar"]); ax[1].axvline(SAR["diff_thr"], color=C["bad"], ls="--")
    ax[1].axvspan(SAR["diff_thr"] * 0.8, SAR["diff_thr"] * 1.2, color=C["bad"], alpha=0.12, label="±20% around my choice")
    ax[1].set_xlabel("radar threshold (dB drop needed)"); ax[1].set_ylabel("flooded area (km²)"); ax[1].legend(); ax[1].set_title("Radar: area vs threshold")
else:
    ax[1].axis("off")
plt.tight_layout(); plt.show()

> 👀 **What to look for:** a *flat* curve around the red dashed line means the answer is stable; a *steep* slope means it is fragile. The next cell turns this into a reportable range.

### 7e. Uncertainty range from a real threshold sweep
We rerun the **whole** pipeline (terrain, sea mask, fusion, speck removal) on a 3 x 3 grid of thresholds (optical and radar each at 80%, 100%, 120% of the chosen value) and report the min to max. Use this range in the report, not a guess.

In [ ]:
mult = (0.8, 1.0, 1.2)
grid = np.full((3, 3), np.nan)
for i_, mo in enumerate(mult):
    for j_, ms in enumerate(mult):
        Mx = fuse_masks(R1, {**OPT, "d_thr": OPT["d_thr"] * mo}, {**SAR, "diff_thr": SAR["diff_thr"] * ms})
        grid[i_, j_] = Mx["flood"].sum() * px_km2
lo, hi, mid = float(np.nanmin(grid)), float(np.nanmax(grid)), float(grid[1, 1])
summary["area_range_km2"] = [round(lo, 3), round(hi, 3)]
fig, ax = plt.subplots(figsize=(6.2, 4.6))
im = ax.imshow(grid, cmap="YlOrRd")
for i_ in range(3):
    for j_ in range(3): ax.text(j_, i_, f"{grid[i_, j_]:.2f}", ha="center", va="center", fontsize=12, fontweight="bold")
ax.set_xticks(range(3)); ax.set_xticklabels([f"{m*SAR['diff_thr']:.1f} dB" for m in mult]); ax.set_xlabel("radar drop threshold")
ax.set_yticks(range(3)); ax.set_yticklabels([f"{m*OPT['d_thr']:.2f}" for m in mult]); ax.set_ylabel("optical change threshold")
ax.set_title("Final flooded area (km²) under threshold changes"); plt.colorbar(im, ax=ax, fraction=0.045)
plt.tight_layout(); plt.show()
print(f"Report this as: {mid:.2f} km² (range {lo:.2f} to {hi:.2f} km² under ±20% threshold changes).")
if mid > 0: print(f"The range spans {(hi - lo) / mid * 100:.0f}% of the central value.")

### 7f. Interactive map (zoom, pan, toggle layers)

In [ ]:
try:
    import folium
    import numpy as np
    from pyproj import Transformer

    t_ll = Transformer.from_crs(f"EPSG:{EPSG}", "EPSG:4326", always_xy=True)

    dx_, dy_ = abs(float(xs[1] - xs[0])), abs(float(ys[1] - ys[0]))
    corners = [
        t_ll.transform(x_, y_)
        for x_ in (xs.min() - dx_ / 2, xs.max() + dx_ / 2)
        for y_ in (ys.min() - dy_ / 2, ys.max() + dy_ / 2)
    ]
    lons, lats = [c[0] for c in corners], [c[1] for c in corners]
    bounds = [[min(lats), min(lons)], [max(lats), max(lons)]]

    north_up = lambda a: a if ys[0] > ys[-1] else a[::-1]

    m = folium.Map(location=[np.mean(lats), np.mean(lons)], zoom_start=11, tiles=None)

    # 1. Esri Satellite Basemap
    folium.TileLayer(
        "https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
        attr="Esri World Imagery",
        name="Satellite basemap (any date)"
    ).add_to(m)

    # 2. Esri World Street Map (Free, no API key required)
    folium.TileLayer(
        "https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}",
        attr="Esri World Street Map",
        name="Street map"
    ).add_to(m)

    # Overlays
    folium.raster_layers.ImageOverlay(north_up(np.dstack([bg_post, np.ones(shape)])), bounds=bounds, opacity=0.9, name="AFTER storm (Sentinel)", show=False).add_to(m)
    folium.raster_layers.ImageOverlay(north_up(np.dstack([bg_pre, np.ones(shape)])), bounds=bounds, opacity=0.9, name="BEFORE storm (Sentinel)", show=False).add_to(m)
    folium.raster_layers.ImageOverlay(north_up(rgba), bounds=bounds, opacity=0.85, name="Detected flood (blue=optical, purple=radar, red=both)").add_to(m)

    folium.LayerControl(collapsed=False).add_to(m)
    m.save("flood_map_interactive.html")
    print("Saved flood_map_interactive.html.")
    display_map = m
except Exception as e:
    display_map = None
    print("Interactive map skipped:", e)

display_map

### 7g. Automatic reality check: should I trust this?
Plain-English traffic lights computed from your own results. ✅ = fine, ⚠️ = look closer, ❌ = something is wrong.

In [ ]:
ev_ = pd.Timestamp(EVENT_DATE)
after_dates = list(S2_post.keys()) + (s1_after if "s1_after" in globals() else ([post_day] if HAVE_RADAR else []))
days_first = min(((pd.Timestamp(d) - ev_).days for d in after_dates), default=None)

sens = None
try:
    pts_ = []
    for f_ in (0.8, 1.2):
        a_ = 0.0
        if HAVE_OPTICAL: a_ += float((optical_flood(mndwi_pre, mndwi_post, **{**OPT, "d_thr": OPT["d_thr"] * f_}) & topo).sum())
        if HAVE_RADAR:   a_ += float((radar_flood(vv_pre_db, vv_post_db, **{**SAR, "diff_thr": SAR["diff_thr"] * f_}) & topo).sum())
        pts_.append(a_)
    mid_ = float((flood_opt.sum() if HAVE_OPTICAL else 0) + (flood_sar.sum() if HAVE_RADAR else 0))
    sens = (max(pts_) - min(pts_)) / mid_ if mid_ > 0 else None
except Exception:
    pass

raw_total = (flood_opt_raw.sum() + flood_sar_raw.sum()); kept_total = (flood_opt.sum() + flood_sar.sum())
metrics = dict(
    fused_km2=summary["flooded_km2_fused"], flooded_frac=float(flood.mean()),
    agree_iou=summary.get("agreement", {}).get("iou_optical_vs_radar") if "agreement" in summary else None,
    terrain_removed_frac=float(1 - kept_total / raw_total) if (HAVE_DEM and raw_total > 0) else None,
    sens_change=sens, days_to_first_after=days_first,
    clear_frac_post=best_post["clear_frac"] if HAVE_OPTICAL else None,
)
if HAVE_OPTICAL and HAVE_RADAR:
    a_, b_ = flood_opt.ravel(), flood_sar.ravel()
    metrics["agree_iou"] = float((a_ & b_).sum() / max((a_ | b_).sum(), 1))
ICON = {"ok": "✅", "warn": "⚠️", "bad": "❌"}
print(f"=== RESULT: about {metrics['fused_km2']} km² newly flooded ===\n")
for level, msg in diagnose(metrics):
    print(ICON[level], msg)

## 8. Who and what was affected? (exposure)
Land cover comes from ESA WorldCover (already loaded). Roads and buildings come from OpenStreetMap. *Wording tip:* these are road segments that **intersect detected inundation** (candidates for inspection), not confirmed damage.

In [ ]:
WC = {10:"Trees",20:"Shrubland",30:"Grassland",40:"Cropland",50:"Built-up",60:"Bare/sparse",70:"Snow/ice",80:"Water",90:"Wetland",95:"Mangroves",100:"Moss"}
exposure = {}
if lc is not None:
    s = pd.Series(lc[flood]).map(WC).value_counts() * px_km2
    exposure["flooded_km2_by_landcover"] = s.round(3).to_dict()
    print("Flooded area by land cover (km²):"); print(s.round(3).to_string())
    ax = s.sort_values().plot.barh(figsize=(7, 3.5), color="#2b6cb0"); ax.set_xlabel("km² flooded"); plt.tight_layout(); plt.savefig("exposure_landcover.png", dpi=150); plt.show()
else:
    print("Land cover not available; skipping.")

In [ ]:
# Roads and buildings from OpenStreetMap (can be slow or empty in remote desert: that is itself a finding)
try:
    import osmnx as ox, geopandas as gpd, rasterio
    from rasterio import features
    from affine import Affine
    dx = float(abs(xs[1] - xs[0])); dy = float(ys[1] - ys[0])
    transform = Affine(dx, 0, xs[0] - dx / 2, 0, dy, ys[0] - dy / 2)   # works for y descending (dy<0) or ascending
    polys = [shape_ for shape_, v in features.shapes(flood.astype("uint8"), mask=flood, transform=transform) if v == 1]
    from shapely.geometry import shape as to_shape
    fgdf = gpd.GeoDataFrame(geometry=[to_shape(p) for p in polys], crs=f"EPSG:{EPSG}")
    flood_union = fgdf.unary_union
    w, s_, e, n = BBOX
    roads = ox.features_from_bbox(bbox=(w, s_, e, n), tags={"highway": True})
    roads = roads[roads.geometry.type.isin(["LineString", "MultiLineString"])].to_crs(EPSG)
    exposure["roads_total_km"] = round(float(roads.length.sum() / 1000), 2)
    exposure["roads_in_flood_km"] = round(float(roads.intersection(flood_union).length.sum() / 1000), 2)
    try:
        b = ox.features_from_bbox(bbox=(w, s_, e, n), tags={"building": True})
        b = b[b.geometry.type.isin(["Polygon", "MultiPolygon"])].to_crs(EPSG)
        exposure["buildings_total"] = int(len(b))
        exposure["buildings_touching_flood"] = int(b.intersects(flood_union).sum())
    except Exception as e2:
        print("No buildings found in this box:", e2)
    print(json.dumps(exposure, indent=2))
    if len(roads):
        hit = roads.intersection(flood_union); hit = hit[~hit.is_empty]
        fig, ax = plt.subplots(figsize=(9, 8))
        fgdf.plot(ax=ax, color=C["optical"], alpha=0.6)
        roads.plot(ax=ax, color="#555555", linewidth=0.6)
        if len(hit): gpd.GeoSeries(hit).plot(ax=ax, color=C["flood_both"], linewidth=2.5)
        legend_box(ax, [("Flooded ground", C["optical"]), ("Roads (OpenStreetMap)", "#555555"), ("Roads inside flood", C["flood_both"])])
        ax.set_title(f"Roads cut or at risk: {exposure.get('roads_in_flood_km', 0)} km of {exposure.get('roads_total_km', 0)} km")
        ax.set_aspect("equal"); ax.set_xlabel("easting (m)"); ax.set_ylabel("northing (m)"); plt.tight_layout(); plt.show()
except Exception as e:
    print("OSM exposure step skipped:", e)
summary["exposure"] = exposure

## 9. Validation: how good is it, really?
Five layers, from cheapest to strongest. Each one answers a different judge question.

**9a. Do the two satellites agree?** (Independent sensors agreeing is evidence, not proof.)

In [ ]:
ev_ = pd.Timestamp(EVENT_DATE)
after_dates = list(S2_post.keys()) + (s1_after if "s1_after" in globals() else ([post_day] if HAVE_RADAR else []))
days_first = min(((pd.Timestamp(d) - ev_).days for d in after_dates), default=None)

sens = None
try:
    pts_ = []
    for f_ in (0.8, 1.2):
        a_ = 0.0
        if HAVE_OPTICAL: a_ += float((optical_flood(mndwi_pre, mndwi_post, **{**OPT, "d_thr": OPT["d_thr"] * f_}) & topo).sum())
        if HAVE_RADAR:   a_ += float((radar_flood(vv_pre_db, vv_post_db, **{**SAR, "diff_thr": SAR["diff_thr"] * f_}) & topo).sum())
        pts_.append(a_)
    mid_ = float((flood_opt.sum() if HAVE_OPTICAL else 0) + (flood_sar.sum() if HAVE_RADAR else 0))
    sens = (max(pts_) - min(pts_)) / mid_ if mid_ > 0 else None
except Exception:
    pass

raw_total = (flood_opt_raw.sum() + flood_sar_raw.sum()); kept_total = (flood_opt.sum() + flood_sar.sum())
metrics = dict(
    fused_km2=summary["flooded_km2_fused"], flooded_frac=float(flood.mean()),
    agree_iou=summary.get("agreement", {}).get("iou_optical_vs_radar") if "agreement" in summary else None,
    terrain_removed_frac=float(1 - kept_total / raw_total) if (HAVE_DEM and raw_total > 0) else None,
    sens_change=sens, days_to_first_after=days_first,
    clear_frac_post=best_post["clear_frac"] if HAVE_OPTICAL else None,
)
if HAVE_OPTICAL and HAVE_RADAR:
    a_, b_ = flood_opt.ravel(), flood_sar.ravel()
    metrics["agree_iou"] = float((a_ & b_).sum() / max((a_ | b_).sum(), 1))
ICON = {"ok": "✅", "warn": "⚠️", "bad": "❌"}
print(f"=== RESULT: about {metrics['fused_km2']} km² newly flooded ===\n")
for level, msg in diagnose(metrics):
    print(ICON[level], msg)

### 9b. Stratified labelling sample (the important one)
The map has four kinds of pixel: **flagged by both sensors**, **optical only**, **radar only**, and **not flagged** (inside the flood-possible zone). We draw a random sample from each and **you label them by eye**. Because we know how big each group is, the maths converts your labels into precision, recall, F1 and IoU **with 95% confidence intervals**, without the distortion of sampling a rare class.

**Steps**
1. Run the cell below. It writes `validation_points_to_label_<event>.csv` and saves **contact sheets** in `label_chips/`: small BEFORE / AFTER / radar-change pictures for every point, numbered to match the CSV.
2. Open each sheet. For each numbered point decide: **1 = really flooded or wet in the AFTER image, 0 = not**. Judge **optical-flagged** points on the optical AFTER image and **radar-flagged** points on the radar change panel (each sensor on the day it actually looked); "dry" points on both.
3. Put the 1/0 values into the `label` column and save the file as `validation_points_labeled_<event>.csv`.

In [ ]:
def zone_mask(R):
    z = R["topo_ok"] & ~R["sea"]
    ref = R["mndwi_post"] if R["have_opt"] else R["vv_post_db"]
    return z & np.isfinite(ref)

def make_validation_sample(R, n=dict(both=25, opt_only=35, rad_only=35, dry=45), seed=42, make_chips=True):
    key = R["ctx"]["key"]; res = R["ctx"]["res"]
    fl, fo, fs = R["flood"], R["opt"], R["sar"]
    masks = {"both": fl & fo & fs, "opt_only": fl & fo & ~fs, "rad_only": fl & ~fo & fs, "dry": zone_mask(R) & ~fl}
    sizes = {k: int(v.sum()) for k, v in masks.items()}
    rng = np.random.default_rng(seed); to_ll = Transformer.from_crs(f"EPSG:{R['ctx']['epsg']}", "EPSG:4326", always_xy=True)
    s2d = max(R["S2_post"], key=lambda d: R["S2_post"][d]["clear_frac"]) if R["S2_post"] else "n/a"
    hint = {"both": f"both: judge S2 AFTER {s2d} and S1 change {R['post_day']}", "opt_only": f"optical: judge S2 AFTER {s2d}",
            "rad_only": f"radar: judge S1 change {R['post_day']}", "dry": "dry: check both"}
    pts = []
    for s, m in masks.items():
        pool = np.argwhere(m)
        if len(pool) == 0: continue
        pick = pool[rng.choice(len(pool), size=min(n[s], len(pool)), replace=False)]
        for r, c in pick:
            lon, lat = to_ll.transform(float(R["xs"][c]), float(R["ys"][r]))
            pts.append(dict(lon=round(lon, 6), lat=round(lat, 6), stratum=s, check_image=hint[s], row=int(r), col=int(c), label=""))
    df = pd.DataFrame(pts).sample(frac=1, random_state=1).reset_index(drop=True); df.insert(0, "id", range(1, len(df) + 1))
    df.to_csv(f"validation_points_to_label_{key}.csv", index=False)
    json.dump(sizes, open(f"validation_strata_sizes_{key}.json", "w"))
    print(f"Wrote validation_points_to_label_{key}.csv with {len(df)} points. Population sizes (pixels):", sizes)
    if make_chips and len(df):
        os.makedirs("label_chips", exist_ok=True)
        if R["have_opt"]: pre_i, post_i = np.clip(R["best_pre"]["rgb"] * 3.5, 0, 1), np.clip(R["best_post"]["rgb"] * 3.5, 0, 1)
        else: pre_i, post_i = gray3(R["vv_pre_db"]), gray3(R["vv_post_db"])
        chg = (R["vv_post_db"] - R["vv_pre_db"]) if R["have_sar"] else None
        half, per_page = 30, 12
        for pg in range(int(np.ceil(len(df) / per_page))):
            sub = df.iloc[pg * per_page:(pg + 1) * per_page]
            fig, axes = plt.subplots(len(sub), 3, figsize=(8.5, 2.45 * len(sub)), squeeze=False)
            for i_, (_, rw) in enumerate(sub.iterrows()):
                r0, c0 = max(rw["row"] - half, 0), max(rw["col"] - half, 0); r1, c1 = r0 + 2 * half, c0 + 2 * half
                for j_, (img, t) in enumerate([(pre_i, "BEFORE"), (post_i, "AFTER")]):
                    axes[i_, j_].imshow(img[r0:r1, c0:c1]); axes[i_, j_].set_title(f"#{rw['id']} {t}", fontsize=8)
                if chg is not None: axes[i_, 2].imshow(chg[r0:r1, c0:c1], cmap="RdBu", vmin=-8, vmax=8); axes[i_, 2].set_title("radar change (blue=darker)", fontsize=8)
                for j_ in range(3):
                    axes[i_, j_].plot(rw["col"] - c0, rw["row"] - r0, "+", color="yellow", ms=10, mew=2); axes[i_, j_].axis("off")
            plt.tight_layout(); plt.savefig(f"label_chips/{key}_sheet_{pg+1:02d}.png", dpi=75); plt.show()
        print("Contact sheets saved in label_chips/ (also shown above). Yellow cross = the point to judge.")
    return df, sizes

val_df, val_sizes = make_validation_sample(R1)

### 9c. Score your labels
Run after you have saved `validation_points_labeled_<event>.csv`. Skip until then.

In [ ]:
def score_validation(R, show=True):
    key = R["ctx"]["key"]; fn = f"validation_points_labeled_{key}.csv"
    if not os.path.exists(fn):
        print(f"Create {fn} first (see Section 9b)."); return None
    v = pd.read_csv(fn).dropna(subset=["label"])
    sizes = json.load(open(f"validation_strata_sizes_{key}.json"))
    if len(v) < 40:
        print("Need at least ~40 labelled points."); return None
    point, ci = strata_estimates(v, sizes)
    names = [("precision", "Precision (fused)"), ("recall", "Recall (fused)"), ("f1", "F1 (fused)"), ("iou", "IoU (fused)"),
             ("precision_optical", "Precision: optical-flagged"), ("precision_radar", "Precision: radar-flagged"), ("precision_both", "Precision: both agree")]
    res_ = {k: dict(value=round(float(point[k]), 3), ci95=[round(ci[k][0], 3), round(ci[k][1], 3)]) for k, _ in names}
    if show:
        fig, ax = plt.subplots(1, 2, figsize=(17, 5))
        vals = [point[k] for k, _ in names]; lo_ = [point[k] - ci[k][0] for k, _ in names]; hi_ = [ci[k][1] - point[k] for k, _ in names]
        cols = [C["ok"] if (x == x and x >= 0.7) else C["warn"] for x in vals]
        ax[0].barh([n for _, n in names][::-1], vals[::-1], xerr=[lo_[::-1], hi_[::-1]], color=cols[::-1], capsize=4)
        ax[0].axvline(0.7, color="grey", ls="--"); ax[0].set_xlim(0, 1.05); ax[0].set_title("Accuracy vs your hand labels (bars = 95% interval)")
        for i_, x in enumerate(vals[::-1]): ax[0].text(min(x + 0.02, 0.98), i_, f"{x:.2f}", va="center")
        rows_ = []
        for s in STRATA:
            g = v[v["stratum"] == s]["label"].astype(int)
            k_, n_ = int(g.sum()), int(len(g)); w = wilson(k_, n_)
            rows_.append(dict(group=s, pixels=sizes[s], labelled=n_, share_really_flooded=round(k_ / n_, 2) if n_ else np.nan, ci95=f"{w[0]:.2f}-{w[1]:.2f}" if n_ else "n/a"))
        tab = pd.DataFrame(rows_); ax[1].axis("off")
        t_ = ax[1].table(cellText=tab.values, colLabels=list(tab.columns), loc="center"); t_.auto_set_font_size(False); t_.set_fontsize(10); t_.scale(1, 1.8)
        ax[1].set_title("What fraction of each group was really flooded?")
        plt.tight_layout(); plt.show()
    print("Report these (zone = flat, low, non-sea ground):")
    for k, n in names: print(f"  {n:30s} {res_[k]['value']:.2f}  (95% interval {res_[k]['ci95'][0]:.2f} to {res_[k]['ci95'][1]:.2f})")
    return res_

val_res = score_validation(R1)
if val_res: summary["validation"] = val_res

> 👀 **How to read it:** *Precision* = of what the map calls flood, how much is real. *Recall* = of the real flood inside the flood-possible zone, how much the map found. The 95% interval shows how much your sample size limits certainty: wide bars mean "label more points". The group table is diagnostic: if "opt_only" has a low real-flood share, optical is the noisy sensor; if "rad_only" does, radar is. Metrics are for the **flood-possible zone** (flat, low, not sea), the realistic area where flooding can occur; say so in the report.

### 9d. Test the "temporal decorrelation" explanation
Claim to test: *radar and optical overlap little because they looked on different days and the water drained in between.* If that is true, radar images taken **closer to the optical date** should overlap the optical map **more** than the day +1 radar image does. We rerun the radar detector on every radar pass in the first two weeks and compare.

In [ ]:
def radar_series(R, max_dates=None):
    ctx = R["ctx"]; ev = pd.Timestamp(ctx["date"]); max_dates = max_dates or TEMPORAL_MAX_DATES
    ext = (ctx["date"], (ev + pd.Timedelta(days=POST_DAYS_EXT)).strftime("%Y-%m-%d"))
    posts = sorted(s1_candidates(ctx, ext), key=lambda i: i.datetime); pres = s1_candidates(ctx, ctx["pre_rng"])
    seen, rows = set(), []
    for p in posts:
        ro, st = p.properties.get("sat:relative_orbit"), p.properties.get("sat:orbit_state"); d = p.datetime.strftime("%Y-%m-%d")
        if (d, ro) in seen: continue
        pre_m = same_orbit(pres, ro, st)
        if not pre_m: continue
        seen.add((d, ro))
        post_items = [i for i in same_orbit(posts, ro, st) if i.datetime.strftime("%Y-%m-%d") == d]
        pdays = sorted({i.datetime.strftime("%Y-%m-%d") for i in pre_m})[-3:]
        try:
            pre_s, _, _ = vv_db_stack([i for i in pre_m if i.datetime.strftime("%Y-%m-%d") in pdays], ctx); post_s, _, _ = vv_db_stack(post_items, ctx)
            m = radar_flood(fit_shape(np.nanmedian(np.stack(pre_s), axis=0), R["shape"]), fit_shape(post_s[0], R["shape"]), **SAR) & R["topo_ok"] & ~R["sea"]
            m = clean_mask(m, MIN_PIXELS)
            rows.append(dict(date=d, days_after_storm=(pd.Timestamp(d) - ev).days, orbit=f"{ro} {st}", radar_km2=round(float(m.sum() * R["px_km2"]), 3),
                             overlap_with_optical_km2=round(float((m & R["opt"]).sum() * R["px_km2"]), 3) if R["have_opt"] else np.nan,
                             iou_with_optical=round(iou(m, R["opt"]), 3) if R["have_opt"] else np.nan))
        except Exception as e:
            print(f" ! radar pass {d} failed: {e}")
        if len(rows) >= max_dates: break
    return pd.DataFrame(rows)

tseries = radar_series(R1) if R1["have_opt"] else pd.DataFrame()
if len(tseries) == 0:
    print("Temporal test not possible (needs optical data and at least one radar pass).")
else:
    opt_days = [(pd.Timestamp(d) - pd.Timestamp(EVENT_DATE)).days for d in R1["S2_post"]]
    display(tseries)
    fig, ax = plt.subplots(1, 2, figsize=(14, 4.4))
    ax[0].bar(tseries["days_after_storm"], tseries["radar_km2"], color=C["radar"], width=0.6, label="radar flood area")
    ax[0].axhline(R1["audit"]["optical_final_km2"], color=C["optical"], ls="--", label="optical flood area")
    for d_ in opt_days: ax[0].axvline(d_, color=C["optical"], alpha=0.5)
    ax[0].set_xlabel("days after storm"); ax[0].set_ylabel("km²"); ax[0].set_title("Radar area by pass (blue lines = optical dates)"); ax[0].legend()
    ax[1].plot(tseries["days_after_storm"], tseries["iou_with_optical"], "-o", color=C["radar"])
    for d_ in opt_days: ax[1].axvline(d_, color=C["optical"], alpha=0.5)
    ax[1].set_xlabel("days after storm"); ax[1].set_ylabel("overlap with optical (IoU)"); ax[1].set_title("Does radar agree more with optical as the dates converge?")
    plt.tight_layout(); plt.show()
    early = tseries.iloc[0]; later = tseries[tseries["days_after_storm"] >= 3]
    if len(later):
        best = later.loc[later["iou_with_optical"].idxmax()]
        if best["iou_with_optical"] >= early["iou_with_optical"] + 0.05:
            verdict = "SUPPORTED: radar overlaps optical noticeably more on a later pass, consistent with the water draining between dates."
        else:
            verdict = "NOT SUPPORTED (or inconclusive): later radar passes do not agree with optical better. The low overlap may reflect detector errors, not just timing; say so in the report."
    else:
        verdict = "INCONCLUSIVE: no radar pass later than day 2 was available, so timing cannot be separated from detector error."
    print("Verdict:", verdict); summary["temporal_test"] = dict(verdict=verdict, table=tseries.to_dict("records"))

> 👀 **How to use the verdict:** write exactly what it says. If supported, it becomes a real finding ("overlap rose from X to Y as the radar date approached the optical date"). If not, replace the "temporal decorrelation, NOT an algorithm error" claim with "the overlap is low; timing may contribute but the test did not confirm it".

### 9e. Does it work somewhere else? (second location, thresholds unchanged)

In [ ]:
def quick_report(R, label=None):
    a = R["audit"]; fl = R["flood"]; px = R["px_km2"]
    bg = np.clip(R["best_post"]["rgb"] * 3.5, 0, 1) if R["have_opt"] else gray3(R["vv_post_db"])
    conf = R["opt"].astype(int) + R["sar"].astype(int); ov = bg.copy()
    ov[fl & (conf <= 1)] = mcolors.to_rgb(C["flood_one"]); ov[fl & (conf == 2)] = mcolors.to_rgb(C["flood_both"])
    fig, ax = plt.subplots(1, 2, figsize=(16, 7))
    ax[0].imshow(bg); ax[0].set_title(f"{label or R['ctx']['label']}: AFTER")
    ax[1].imshow(ov); ax[1].set_title(f"Detected flood: {a['fused_final_km2']} km²")
    legend_box(ax[1], [("Both satellites agree", C["flood_both"]), ("One satellite only", C["flood_one"])], title="Confidence")
    for a_ in ax: a_.axis("off"); scalebar(a_, R["ctx"]["res"])
    plt.tight_layout(); plt.show()
    m = dict(fused_km2=a["fused_final_km2"], flooded_frac=float(fl.mean()), agree_iou=iou(R["opt"], R["sar"]) if (R["have_opt"] and R["have_sar"]) else None,
             terrain_removed_frac=None, sens_change=None, clear_frac_post=R["best_post"]["clear_frac"] if R["have_opt"] else None,
             days_to_first_after=min(((pd.Timestamp(d) - pd.Timestamp(R["ctx"]["date"])).days for d in list(R["S2_post"]) + ([R["post_day"]] if R["post_day"] else [])), default=None))
    for level, msg in diagnose(m): print({"ok": "✅", "warn": "⚠️", "bad": "❌"}[level], msg)

R2 = None
try:
    R2 = process_event(SECONDARY)
    quick_report(R2)
    cmp_ = pd.DataFrame({"": ["optical km²", "radar km²", "overlap km²", "FINAL km²", "sea removed km²", "specks removed km²"],
                         PRIMARY: [R1["audit"][k] for k in ("optical_final_km2", "radar_final_km2", "overlap_km2", "fused_final_km2", "sea_removed_km2", "removed_small_blobs_km2")],
                         SECONDARY: [R2["audit"][k] for k in ("optical_final_km2", "radar_final_km2", "overlap_km2", "fused_final_km2", "sea_removed_km2", "removed_small_blobs_km2")]})
    display(cmp_); summary["secondary"] = dict(event=SECONDARY, audit=R2["audit"])
except Exception as e:
    print("Second location could not be processed:", e)

> 👀 **What to look for:** the same thresholds, a different place. If the red/orange again follows wadi channels and the diagnostics are not alarming, the method generalises. Optionally label this one too: run `make_validation_sample(R2)` then `score_validation(R2)`.

## 10. Predicting floods BEFORE they happen: flood susceptibility machine learning

### What we can and cannot claim
- **Can:** learn *where* water goes during a hard rainfall from things known **before** the storm: terrain, drainage, land cover and pre-storm imagery. That produces a **risk map** that could be made ahead of any storm.
- **Cannot (yet):** forecast *when* and *how large*. A rainfall-to-flood-extent model needs many labelled storms; we have one or two. Instead, the design couples this risk map to a rainfall forecast trigger (diagram below).
- **Honest caveat:** the "flooded" labels come from our own detector, not ground truth, from a single storm. The strongest evidence is the **transfer test**: train in one place, predict a different place.

### The model, in plain English
For every pixel we compute 13 features (how high, how steep, how much land drains into it, how far above the nearest wadi, ...). A **gradient-boosted decision-tree model** learns which combinations were flooded. It is evaluated with **spatial cross-validation**: the map is cut into 2 km blocks, and each block is predicted by a model that never saw it.

**Why this model and not deep learning?** One storm gives limited labels; tree models need little data, train in seconds, and show which features matter. A deep network (e.g. a U-Net on terrain patches) is listed as future work once more storms are labelled.

In [ ]:
def build_features(R):
    T = terrain_features(R["dem"], R["ctx"]["res"], ML["drain_km2"]); res = R["ctx"]["res"]; shp = R["shape"]
    sea = R["sea"]
    dist_coast = ndi.distance_transform_edt(~sea, sampling=res) if sea.any() else np.full(shp, np.nan)
    nan = np.full(shp, np.nan, dtype="float32")
    layers = dict(elevation_m=T["elev"], slope_deg=T["slope"], curvature=T["curv"], rel_elev_500m=T["rel500"], rel_elev_1500m=T["rel1500"],
                  upstream_area_log10_km2=T["log_area"], topographic_wetness=T["twi"], dist_to_channel_m=T["dist_drain"], hand_m=T["hand"],
                  dist_to_coast_m=dist_coast, land_cover=R["lc"].astype("float32") if R["lc"] is not None else nan,
                  ndvi_pre=R["ndvi_pre"] if R["ndvi_pre"] is not None else nan, mndwi_pre=R["mndwi_pre"] if R["mndwi_pre"] is not None else nan)
    names = list(layers)
    F = np.stack([np.asarray(layers[k], dtype="float32") for k in names], axis=-1)
    return F, names, T

assert R1["have_dem"], "Susceptibility modelling needs the DEM (terrain) to have loaded."
F1, FEATURES, T1 = build_features(R1)
y1 = R1["flood"]; domain1 = ~R1["sea"] & np.isfinite(T1["elev"]); zone1 = domain1 & R1["topo_ok"]
print(f"{len(FEATURES)} features: {FEATURES}")
print(f"Flood pixels: {int((y1 & domain1).sum()):,} | flood-possible zone: {int(zone1.sum()):,} px | base rate in zone: {(y1 & zone1).sum() / max(zone1.sum(), 1) * 100:.2f}%")

### 10a. What the model sees (key features, flood outline in red)

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(19, 11))
panels = [("Upstream area (log10 km²): how much land drains here", T1["log_area"], "viridis", None),
          ("Height above nearest channel, HAND (m)", np.clip(T1["hand"], 0, 30), "magma_r", None),
          ("Topographic wetness index", T1["twi"], "Blues", None),
          ("Slope (deg)", np.clip(T1["slope"], 0, 30), "gray_r", None),
          ("Elevation (m)", T1["elev"], "terrain", None),
          ("Distance to nearest channel (m)", np.clip(T1["dist_drain"], 0, 2000), "cividis_r", None)]
for a_, (t, arr, cm, _) in zip(ax.ravel(), panels):
    arr = np.where(R1["sea"], np.nan, arr)
    im = a_.imshow(arr, cmap=cm); a_.contour(y1.astype(float), levels=[0.5], colors="red", linewidths=0.7)
    a_.set_title(t, fontsize=11); a_.axis("off"); plt.colorbar(im, ax=a_, fraction=0.035)
plt.tight_layout(); plt.show()

> 👀 **What to look for:** the red flood outline should hug the bright wadi network in the upstream-area map and sit in the **low HAND** (dark) zones. If so, the physics supports the detections, and the model has something real to learn. The white gaps are the sea (excluded).

### 10b. Train and test with spatial cross-validation

In [ ]:
folds1 = block_folds(R1["shape"], RES, ML["block_m"], ML["folds"])
P1, models1 = oof_susceptibility(F1, y1, domain1, folds1, ML["max_pos"], ML["neg_ratio"])
yb = y1.astype(bool)
hand_i, area_i = FEATURES.index("hand_m"), FEATURES.index("upstream_area_log10_km2"); twi_i = FEATURES.index("topographic_wetness")
scores = {"Model (out-of-fold)": P1, "Baseline: low HAND": -F1[..., hand_i], "Baseline: big upstream area": F1[..., area_i], "Baseline: wetness index": F1[..., twi_i]}
rows = []
for nm, sc in scores.items():
    for zn, mk in [("flood-possible zone", zone1), ("whole land area", domain1)]:
        m_ = score_metrics(sc, yb, mk); rows.append(dict(model=nm, evaluated_on=zn, AUC=round(m_["auc"], 3), AP=round(m_["ap"], 3), base_rate=round(m_["base_rate"], 4)))
ml_tab = pd.DataFrame(rows); display(ml_tab)
primary_m = score_metrics(P1, yb, zone1)
summary["ml"] = dict(primary_oof_zone=primary_m, table=ml_tab.to_dict("records"))
print(f"Spatial-CV result in the flood-possible zone: AUC {primary_m['auc']:.3f}, average precision {primary_m['ap']:.3f} (a random model would score AP = {primary_m['base_rate']:.3f}).")

> 👀 **How to read the table:** *AUC* near 0.5 = guessing, above 0.8 = useful. *AP* must be judged against **base_rate** (the share of the land that flooded): AP many times larger than base_rate is real skill. The **flood-possible zone** is the honest test; "whole land area" is flattered by easy cases (mountains are obviously dry). If the model barely beats the simple baselines, say so: it means terrain alone already explains the pattern, which is a valid and interesting finding.

### 10c. The risk map and how well it ranks places

In [ ]:
qs = np.nanquantile(P1[zone1], [0.80, 0.95, 0.99])
risk = np.zeros(R1["shape"], dtype=np.uint8)
risk[zone1] = 1
risk[zone1 & (P1 >= qs[0])] = 2; risk[zone1 & (P1 >= qs[1])] = 3; risk[zone1 & (P1 >= qs[2])] = 4
RISK_NAME = {0: "Not flood-prone (steep/high/sea)", 1: "Low", 2: "Moderate", 3: "High", 4: "Very high"}
RISK_COL = {0: "#d9d9d9", 1: "#ffffb2", 2: "#fecc5c", 3: "#fd8d3c", 4: "#bd0026"}
cm_risk = ListedColormap([RISK_COL[k] for k in range(5)])
tab = []
for k in range(1, 5):
    m_ = risk == k
    tab.append(dict(risk_class=RISK_NAME[k], share_of_zone_pct=round(100 * m_.sum() / zone1.sum(), 1), area_km2=round(m_.sum() * px_km2, 2),
                    share_of_actual_flood_pct=round(100 * (m_ & y1).sum() / max(y1.sum(), 1), 1)))
risk_tab = pd.DataFrame(tab); display(risk_tab)
fr, gn = gain_curve(P1, yb, zone1)
top5 = float(np.interp(0.05, fr, gn)); top20 = float(np.interp(0.20, fr, gn))
summary["ml"].update(risk_table=risk_tab.to_dict("records"), flood_caught_top5pct=round(top5, 3), flood_caught_top20pct=round(top20, 3))

fig, ax = plt.subplots(1, 3, figsize=(22, 6.4), gridspec_kw={"width_ratios": [1.4, 1.4, 1]})
im = ax[0].imshow(np.where(domain1, P1, np.nan), cmap="inferno", vmin=0, vmax=1); ax[0].contour(y1.astype(float), levels=[0.5], colors="cyan", linewidths=0.6)
ax[0].set_title("Flood susceptibility score (out-of-fold, cyan = storm flood)"); ax[0].axis("off"); plt.colorbar(im, ax=ax[0], fraction=0.04, label="relative risk score (0-1)"); scalebar(ax[0], RES)
ax[1].imshow(risk, cmap=cm_risk, vmin=0, vmax=4); ax[1].set_title("Risk classes"); ax[1].axis("off")
legend_box(ax[1], [(RISK_NAME[k], RISK_COL[k]) for k in range(5)], loc="lower left"); scalebar(ax[1], RES)
ax[2].plot(fr * 100, gn * 100, color=C["bad"], lw=2.5, label="model"); ax[2].plot([0, 100], [0, 100], "--", color="grey", label="random")
ax[2].axvline(5, color="grey", alpha=0.4); ax[2].scatter([5], [top5 * 100], color=C["bad"], zorder=5)
ax[2].annotate(f"flag the riskiest 5% of the land\n-> catches {top5*100:.0f}% of the flood", xy=(5, top5 * 100), xytext=(18, max(top5 * 100 - 25, 8)), arrowprops=dict(arrowstyle="->"))
ax[2].set_xlabel("% of flood-possible land flagged (riskiest first)"); ax[2].set_ylabel("% of the real flood caught"); ax[2].set_title("Gain curve"); ax[2].legend(loc="lower right")
plt.tight_layout(); plt.savefig("risk_dashboard.png", dpi=150); plt.show()
print(f"Flagging the riskiest 5% of flood-possible land catches {top5*100:.0f}% of the storm flood; the riskiest 20% catches {top20*100:.0f}%.")

> 👀 **What to look for:** the "Very high" and "High" classes should hold a **much larger share of the real flood than their share of the land** (compare the two percentage columns). That gap is the model's value: it tells authorities where to focus inspection and preparation. The gain curve is the one-picture version: the further it bows above the diagonal, the better. *The score is a relative ranking, not the probability of flooding.*

### 10d. The hardest test: predict a **different place** (transfer)
Train on the primary location only, then predict the second location without retraining.

In [ ]:
transfer = None
if R2 is not None and R2["have_dem"] and R2["flood"].sum() > 0:
    F2, _, T2 = build_features(R2); y2 = R2["flood"]; domain2 = ~R2["sea"] & np.isfinite(T2["elev"]); zone2 = domain2 & R2["topo_ok"]
    X, t = sample_xy(F1, y1, domain1, folds1, f_excl=None, max_pos=ML["max_pos"], neg_ratio=ML["neg_ratio"])
    model_full = make_model().fit(X, t, sample_weight=balanced_w(t))
    P2 = predict_chunks(model_full, F2, domain2); y2b = y2.astype(bool)
    sc2 = {"Model (trained on primary)": P2, "Baseline: low HAND": -F2[..., hand_i], "Baseline: big upstream area": F2[..., area_i], "Baseline: wetness index": F2[..., twi_i]}
    rows = []
    for nm, sc in sc2.items():
        m_ = score_metrics(sc, y2b, zone2); rows.append(dict(model=nm, AUC=round(m_["auc"], 3), AP=round(m_["ap"], 3), base_rate=round(m_["base_rate"], 4)))
    transfer = pd.DataFrame(rows); display(transfer)
    fr2, gn2 = gain_curve(P2, y2b, zone2)
    summary["ml"]["transfer"] = dict(table=transfer.to_dict("records"), flood_caught_top5pct=round(float(np.interp(0.05, fr2, gn2)), 3))
    fig, ax = plt.subplots(1, 3, figsize=(21, 5.8))
    for nm, sc, st in [("Model: out-of-fold (same place)", (P1, yb, zone1), "-"), ("Model: transfer (new place)", (P2, y2b, zone2), "-"),
                       ("HAND baseline: new place", (-F2[..., hand_i], y2b, zone2), "--")]:
        s_, y_, m_ = sc; mk = m_ & np.isfinite(s_)
        fp, tp, _ = roc_curve(y_[mk], s_[mk]); pr, rc, _ = precision_recall_curve(y_[mk], s_[mk])
        col = {"Model: out-of-fold (same place)": C["optical"], "Model: transfer (new place)": C["bad"], "HAND baseline: new place": "#777777"}[nm]
        ax[0].plot(fp, tp, st, color=col, label=nm); ax[1].plot(rc, pr, st, color=col, label=nm)
    ax[0].plot([0, 1], [0, 1], ":", color="grey"); ax[0].set_xlabel("false positive rate"); ax[0].set_ylabel("true positive rate"); ax[0].set_title("ROC (flood-possible zone)"); ax[0].legend(fontsize=9)
    ax[1].set_xlabel("recall"); ax[1].set_ylabel("precision"); ax[1].set_title("Precision-recall"); ax[1].legend(fontsize=9)
    im = ax[2].imshow(np.where(domain2, P2, np.nan), cmap="inferno", vmin=0, vmax=1); ax[2].contour(y2.astype(float), levels=[0.5], colors="cyan", linewidths=0.6)
    ax[2].set_title("Prediction for the NEW location (cyan = its storm flood)"); ax[2].axis("off"); plt.colorbar(im, ax=ax[2], fraction=0.04)
    plt.tight_layout(); plt.show()
    t_auc = float(transfer.loc[0, "AUC"]); b_auc = float(transfer.loc[1, "AUC"])
    print(f"Transfer AUC {t_auc:.2f} vs. low-HAND baseline {b_auc:.2f}.",
          "The model generalises and beats the simple baseline." if t_auc >= 0.75 and t_auc > b_auc else
          ("The model works but does not beat the simple physical baseline: report that honestly." if t_auc >= 0.65 else "Transfer is weak: report that the model has not been shown to generalise and needs more storms."))
else:
    print("Transfer test skipped (second location not available or has no detected flood).")

> 👀 **How to use this:** this is your strongest ML evidence, because the model never saw the second location. Quote the transfer AUC next to the baseline and say which wins. A modest transfer score with an honest explanation scores better with judges than an inflated same-place score.

### 10e. What drives the risk? (feature importance)

In [ ]:
try:
    fpick = max(models1, key=lambda f: int((y1 & zone1 & (folds1 == f)).sum()))
    te = np.flatnonzero((zone1 & (folds1 == fpick)).ravel()); rng_ = np.random.default_rng(0)
    pos_te = te[y1.ravel()[te]]; neg_te = te[~y1.ravel()[te]]
    idx = np.r_[rng_.choice(pos_te, min(len(pos_te), 3000), replace=False), rng_.choice(neg_te, min(len(neg_te), 9000), replace=False)]
    Xte, yte = F1.reshape(-1, F1.shape[-1])[idx], y1.ravel()[idx].astype(int)
    pi = permutation_importance(models1[fpick], Xte, yte, n_repeats=3, scoring="average_precision", random_state=0, n_jobs=1)
    imp = pd.Series(pi.importances_mean, index=FEATURES).sort_values()
    fig, ax = plt.subplots(figsize=(9, 5.5)); ax.barh(imp.index, imp.values, color=C["optical"]); ax.set_xlabel("drop in average precision when this feature is scrambled")
    ax.set_title("Which evidence matters most? (held-out spatial block)"); plt.tight_layout(); plt.show()
    summary["ml"]["top_features"] = list(imp.sort_values(ascending=False).index[:3])
    print("Top drivers:", ", ".join(summary["ml"]["top_features"]))
except Exception as e:
    print("Feature importance skipped:", e)

> 👀 **What to look for:** physically sensible drivers (upstream area, height above channel, distance to channel, slope) mean the model learned hydrology, not a quirk of the data. If a pre-storm image feature dominates, check it isn't acting as a location fingerprint.

### 10f. From map to action: priority road crossings to inspect or protect *before* the next storm
Roads from OpenStreetMap are walked every 100 m; the points with the highest susceptibility (and at least 1 km apart) form a short inspection list: candidate sites for culverts, signage or closures.

In [ ]:
def priority_crossings(roads_gdf, P, xs_, ys_, thr, k=15, spacing_m=1000, step_m=100):
    pts = []
    for geom in roads_gdf.geometry:
        parts = [geom] if geom.geom_type == "LineString" else list(getattr(geom, "geoms", []))
        for g in parts:
            for d in np.arange(0, g.length, step_m):
                p = g.interpolate(d); pts.append((p.x, p.y))
    if not pts: return pd.DataFrame()
    xy = np.array(pts); col = np.round((xy[:, 0] - xs_[0]) / (xs_[1] - xs_[0])).astype(int); row = np.round((xy[:, 1] - ys_[0]) / (ys_[1] - ys_[0])).astype(int)
    ok = (row >= 0) & (row < P.shape[0]) & (col >= 0) & (col < P.shape[1]); xy, row, col = xy[ok], row[ok], col[ok]
    sc = P[row, col]; keep = np.isfinite(sc) & (sc >= thr); xy, sc = xy[keep], sc[keep]
    order = np.argsort(-sc); chosen = []
    for i_ in order:
        if all(np.hypot(*(xy[i_] - xy[j_])) >= spacing_m for j_ in chosen):
            chosen.append(i_)
        if len(chosen) >= k: break
    to_ll = Transformer.from_crs(f"EPSG:{EPSG}", "EPSG:4326", always_xy=True)
    out = []
    for rank, i_ in enumerate(chosen, 1):
        lon, lat = to_ll.transform(float(xy[i_, 0]), float(xy[i_, 1])); out.append(dict(rank=rank, lat=round(lat, 5), lon=round(lon, 5), risk_score=round(float(sc[i_]), 3), x_utm=xy[i_, 0], y_utm=xy[i_, 1]))
    return pd.DataFrame(out)

crossings = pd.DataFrame()
if "roads" in globals() and len(roads):
    crossings = priority_crossings(roads, P1, xs, ys, thr=qs[1])
if len(crossings):
    display(crossings[["rank", "lat", "lon", "risk_score"]]); crossings.drop(columns=["x_utm", "y_utm"]).to_csv("priority_crossings.csv", index=False)
    fig, ax = plt.subplots(figsize=(9, 8)); ax.imshow(risk, cmap=cm_risk, vmin=0, vmax=4, alpha=0.7)
    dx_ = xs[1] - xs[0]; dy_ = ys[1] - ys[0]
    for g in roads.geometry.iloc[:3000]:
        for part in ([g] if g.geom_type == "LineString" else list(getattr(g, "geoms", []))):
            xx, yy = part.xy; ax.plot((np.array(xx) - xs[0]) / dx_, (np.array(yy) - ys[0]) / dy_, color="#333333", lw=0.5)
    ax.scatter((crossings["x_utm"] - xs[0]) / dx_, (crossings["y_utm"] - ys[0]) / dy_, s=120, c="white", edgecolor="black", zorder=5)
    for _, r_ in crossings.iterrows(): ax.text((r_["x_utm"] - xs[0]) / dx_, (r_["y_utm"] - ys[0]) / dy_, str(int(r_["rank"])), ha="center", va="center", fontsize=8, fontweight="bold", zorder=6)
    ax.set_xlim(0, R1["shape"][1]); ax.set_ylim(R1["shape"][0], 0); ax.axis("off"); ax.set_title("Top road sites to inspect before the next storm (numbered)")
    legend_box(ax, [(RISK_NAME[k], RISK_COL[k]) for k in range(1, 5)], loc="lower left"); plt.tight_layout(); plt.show()
    summary["ml"]["priority_sites"] = len(crossings)
else:
    print("No roads available from OpenStreetMap in this box, so no priority list was made (an honest finding for remote desert areas: say that mapped road data is sparse).")

### 10g. Rainfall context and the early-warning design
Rainfall archives show how exceptional the storm was. (Open-Meteo reanalysis, a coarse ~10 km product; use it for context, not as a measurement.) The diagram shows how the pieces fit into an operational system. **Solid boxes are built in this notebook; dashed boxes are proposed.**

In [ ]:
def fetch_rain(lat, lon, start, end):
    import requests
    r = requests.get("https://archive-api.open-meteo.com/v1/archive", timeout=30,
                     params=dict(latitude=lat, longitude=lon, start_date=start, end_date=end, daily="precipitation_sum", timezone="Africa/Cairo"))
    r.raise_for_status(); j = r.json()["daily"]
    return pd.DataFrame({"date": pd.to_datetime(j["time"]), "precip_mm": j["precipitation_sum"]})

try:
    ev = pd.Timestamp(EVENT_DATE); c_lat, c_lon = (BBOX[1] + BBOX[3]) / 2, (BBOX[0] + BBOX[2]) / 2
    rain = fetch_rain(c_lat, c_lon, (ev - pd.Timedelta(days=10)).strftime("%Y-%m-%d"), (ev + pd.Timedelta(days=5)).strftime("%Y-%m-%d"))
    fig, ax = plt.subplots(figsize=(11, 3.4)); ax.bar(rain["date"], rain["precip_mm"], color="#4a90c2")
    ax.axvline(ev, color="black", lw=2); ax.text(ev, ax.get_ylim()[1] * 0.92, " storm", fontweight="bold")
    ax.set_ylabel("daily rain (mm)"); ax.set_title("Rainfall around the event (reanalysis, ~10 km grid)"); plt.tight_layout(); plt.show()
    summary["rain_peak_mm"] = round(float(rain["precip_mm"].max()), 1); print("Peak daily rainfall in the window:", summary["rain_peak_mm"], "mm")
except Exception as e:
    print("Rainfall context skipped:", e)

from matplotlib.patches import FancyBboxPatch
def draw_pipeline():
    fig, ax = plt.subplots(figsize=(15, 5.6)); ax.axis("off"); ax.set_xlim(0, 15); ax.set_ylim(0, 6)
    def box(x, y, w, h, text, built=True, color="#dbeafe"):
        ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.08", fc=color if built else "white", ec="black", lw=1.6, ls="-" if built else "--"))
        ax.text(x + w / 2, y + h / 2, text, ha="center", va="center", fontsize=9.5)
    def arr(x0, y0, x1, y1): ax.annotate("", xy=(x1, y1), xytext=(x0, y0), arrowprops=dict(arrowstyle="-|>", lw=1.6))
    ax.text(0.1, 5.65, "BEFORE THE STORM (prediction)", fontweight="bold"); ax.text(0.1, 2.55, "AFTER THE STORM (detection)", fontweight="bold")
    box(0.1, 3.9, 2.8, 1.4, "Static inputs\nDEM · land cover\npre-storm imagery"); box(3.7, 3.9, 3.0, 1.4, "Susceptibility model\ngradient boosting\nspatial cross-validation", color="#fde68a")
    box(7.5, 3.9, 2.6, 1.4, "Risk map\n+ priority road sites", color="#fecaca"); box(10.9, 3.9, 2.0, 1.4, "Rainfall forecast\ntrigger (EMA)", built=False)
    box(13.1, 3.9, 1.8, 1.4, "Pre-alert to\ncrisis rooms", built=False)
    arr(2.9, 4.6, 3.7, 4.6); arr(6.7, 4.6, 7.5, 4.6); arr(10.1, 4.6, 10.9, 4.6); arr(12.9, 4.6, 13.1, 4.6)
    box(0.1, 0.5, 2.8, 1.4, "Sentinel-1 radar\nSentinel-2 optical"); box(3.7, 0.5, 3.0, 1.4, "Change detection\n+ terrain + sea masks"); box(7.5, 0.5, 2.6, 1.4, "Flood extent map\n+ exposure (km², roads)", color="#fecaca")
    box(10.9, 0.5, 2.0, 1.4, "Validation\n(labels, CI)"); box(13.1, 0.5, 1.8, 1.4, "New labels ->\nretrain model", built=False)
    arr(2.9, 1.2, 3.7, 1.2); arr(6.7, 1.2, 7.5, 1.2); arr(10.1, 1.2, 10.9, 1.2); arr(12.9, 1.2, 13.1, 1.2)
    arr(14.0, 1.9, 5.2, 3.9); ax.text(9.3, 2.95, "every storm adds labelled data", fontsize=9, style="italic", rotation=18)
    ax.set_title("Detect + predict: how the notebook fits an operational early-warning system (dashed = proposed)", fontsize=12, fontweight="bold")
    plt.tight_layout(); plt.savefig("architecture.png", dpi=150); plt.show()
draw_pipeline()

> 👀 **What to say about it:** every storm detected by satellites becomes new training data for the susceptibility model, so the system improves with each event. The rainfall trigger and alert are the **proposed** operational pieces; the model, the risk map and the priority list are built and tested here.

## 11. One-page dashboard and saving everything

In [ ]:
# A single picture summarising everything: put this on a slide.
from matplotlib.gridspec import GridSpec
fig = plt.figure(figsize=(18, 9))
gs = GridSpec(2, 3, figure=fig, width_ratios=[1.6, 0.8, 0.8])
axm = fig.add_subplot(gs[:, 0]); axk = fig.add_subplot(gs[0, 1:]); axb = fig.add_subplot(gs[1, 1]); axl = fig.add_subplot(gs[1, 2])
axm.imshow(bg_post); axm.imshow(rgba); axm.axis("off"); scalebar(axm, RES)
legend_box(axm, [(CLS_NAME[k], CLS_COL[k]) for k in (1, 2, 3)])
axm.set_title(f"{EV['label']}  |  storm {EVENT_DATE}", fontsize=14)
axk.axis("off")
lines = [f"{summary['flooded_km2_fused']} km² newly flooded", f"≈ {summary['flooded_km2_fused']*1e6/7140:,.0f} football pitches"]
lines += [f"Sensors: {' + '.join(summary['sensors_used'])}", f"Both sensors agree on {summary['both_sensors_agree_km2']} km²"]
ex = summary.get("exposure", {})
if "roads_in_flood_km" in ex: lines.append(f"Roads in flood: {ex['roads_in_flood_km']} km of {ex['roads_total_km']} km")
if "buildings_touching_flood" in ex: lines.append(f"Buildings touching flood: {ex['buildings_touching_flood']} of {ex['buildings_total']}")
if "validation" in summary: lines.append(f"Accuracy vs hand labels: F1 {summary['validation']['f1']}, precision {summary['validation']['precision']}, recall {summary['validation']['recall']}")
else: lines.append("Accuracy: not yet validated (Section 9b)")
for i_, t_ in enumerate(lines):
    axk.text(0.0, 0.95 - i_ * 0.14, t_, fontsize=17 if i_ == 0 else 13, fontweight="bold" if i_ == 0 else "normal", transform=axk.transAxes, va="top")
axb.barh(list(areas.keys()), list(areas.values()), color=[CLS_COL[k] for k in (1, 2, 3)]); axb.set_xlabel("km²"); axb.set_title("Detected by")
lcs = exposure.get("flooded_km2_by_landcover")
if lcs:
    s_ = pd.Series(lcs).sort_values(); axl.barh(s_.index, s_.values, color="#2b6cb0"); axl.set_xlabel("km² flooded"); axl.set_title("Land affected")
else:
    axl.axis("off"); axl.text(0.5, 0.5, "land cover not available", ha="center")
plt.tight_layout(); plt.savefig("flood_dashboard.png", dpi=150); plt.show()

In [ ]:
# ---- save rasters and tables ----
try:
    import xarray as xr, rioxarray
    def save_tif(arr, name, dtype):
        da = xr.DataArray(arr.astype(dtype), coords={"y": ys, "x": xs}, dims=("y", "x")).rio.write_crs(f"EPSG:{EPSG}"); da.rio.to_raster(name)
    save_tif(flood, "flood_extent.tif", "uint8")
    save_tif(np.nan_to_num(P1, nan=-1), "flood_susceptibility.tif", "float32")
    save_tif(risk, "risk_classes.tif", "uint8")
    print("Saved flood_extent.tif, flood_susceptibility.tif, risk_classes.tif (open in QGIS / Google Earth)")
except Exception as e:
    print("GeoTIFF export skipped:", e)
summary["thresholds"] = dict(optical=OPT, radar=SAR, terrain=dict(max_slope_deg=MAX_SLOPE_DEG, max_rel_elev_m=MAX_REL_ELEV_M), sea_buffer_m=SEA_BUFFER_M, min_pixels=MIN_PIXELS, fusion=FUSION)
with open("flood_summary.json", "w") as f_: json.dump(summary, f_, indent=2, default=str)
print("Saved flood_summary.json")

# ---- plain-language numbers for the slides ----
a = audit; L = []
L.append(f"# Results to paste into the report and deck: {EV['label']} ({EVENT_DATE})\n")
L.append("## Detection")
L.append(f"- Final flooded area: **{a['fused_final_km2']} km²** (about {a['fused_final_km2']*1e6/7140:,.0f} football pitches), range **{summary.get('area_range_km2', ['?', '?'])[0]} to {summary.get('area_range_km2', ['?', '?'])[1]} km²** under ±20% threshold changes.")
L.append(f"- Optical {a['optical_final_km2']} km², radar {a['radar_final_km2']} km², overlap {a['overlap_km2']} km². Sea/coast mask removed {a['sea_removed_km2']} km² of detections; terrain mask and speck removal as in the audit chart (Section 6).")
if "exposure" in summary and summary["exposure"]:
    ex = summary["exposure"]
    if "roads_in_flood_km" in ex: L.append(f"- Road segments intersecting detected flooding: {ex['roads_in_flood_km']} km of {ex['roads_total_km']} km mapped (candidates for inspection, not confirmed damage).")
    if "buildings_total" in ex: L.append(f"- Mapped OSM buildings touching the flood: {ex.get('buildings_touching_flood', 0)} of {ex['buildings_total']} (OSM building coverage in desert areas is sparse).")
if "validation" in summary:
    v_ = summary["validation"]; L.append("\n## Validation (hand-labelled stratified sample, flood-possible zone)")
    for k, lab in [("precision", "Precision"), ("recall", "Recall"), ("f1", "F1"), ("iou", "IoU")]: L.append(f"- {lab}: {v_[k]['value']} (95% interval {v_[k]['ci95'][0]} to {v_[k]['ci95'][1]})")
if "temporal_test" in summary: L.append("\n## Temporal-gap test\n- " + summary["temporal_test"]["verdict"])
if "secondary" in summary: L.append(f"\n## Second location ({SECONDARY}, thresholds unchanged)\n- Final flooded area: {summary['secondary']['audit']['fused_final_km2']} km².")
if "ml" in summary:
    m_ = summary["ml"]; L.append("\n## Prediction (flood susceptibility model)")
    L.append(f"- Spatial cross-validation in the flood-possible zone: AUC {m_['primary_oof_zone']['auc']:.2f}, average precision {m_['primary_oof_zone']['ap']:.3f} (random would be {m_['primary_oof_zone']['base_rate']:.3f}).")
    L.append(f"- Flagging the riskiest 5% of flood-possible land catches {m_['flood_caught_top5pct']*100:.0f}% of the storm flood; the riskiest 20% catches {m_['flood_caught_top20pct']*100:.0f}%.")
    if "transfer" in m_: L.append(f"- Transfer to a different location (never seen in training): see table in Section 10d; the riskiest 5% catches {m_['transfer']['flood_caught_top5pct']*100:.0f}% of that location's flood.")
    if "top_features" in m_: L.append(f"- Main drivers: {', '.join(m_['top_features'])}.")
L.append("\n## Say these limitations out loud\n- Labels for the model come from our own detector (single storm), not independent ground truth; the score is a relative ranking, not a probability.\n- Satellite passes are every ~5-6 days, so maps show water present at the image time, not the storm peak.\n- Radar can be fooled by sand and shadows; optical by turbid water and coast; both are reduced, not eliminated.")
open("results_for_slides.md", "w").write("\n".join(L)); print("Saved results_for_slides.md\n"); print("\n".join(L))

## 12. Limitations (say these out loud; judges respect it)
- **Timing gap:** satellites pass every ~5-6 days; maps show water present on the image date, not the storm's peak. Section 9d tests how much this matters.
- **Sand, shadows and coast:** reduced by change detection, the terrain mask, the sea mask and the optical cross-check, but not to zero.
- **Cities:** radar is unreliable among buildings and optical is blocked by storm clouds; street-level urban flooding (e.g. Alexandria) is poorly captured. For cities the susceptibility approach (Section 10) is the better product.
- **Labels for the ML are detections, not ground truth**, from one storm. The transfer test is the guard against overfitting; more labelled storms (2020, 2024, 2025, 2026) are the main route to a real forecast.
- **Risk score is a ranking, not a probability.** It says where water is likely to go, not how much rain will fall.
- **Resolution:** 20 m pixels (10 m possible). Narrow wadi channels may be under-counted.
- **Thresholds** are starting values; the uncertainty range (Section 7d) and validation (Section 9) quantify how far to trust them.

## 13. The product, not just the model
**Problem.** After desert flash floods, authorities in the Red Sea, Sinai and Upper Egypt must decide where to send help and which roads are cut, with little overview for days. Today they rely on field reports and patrols; satellite flood products exclude arid areas.
**Users.** Governorate crisis rooms, flood-protection and irrigation authorities, civil defence, insurers.
**Delivery.** (1) *Before:* a susceptibility risk map and a priority list of road sites, refreshed when new terrain or storm data arrives; with a rainfall-forecast trigger, a pre-alert. (2) *After:* a rapid flood-extent report within about a day of the first clear satellite pass: map, flooded km² with an uncertainty range, roads and land affected, and a confidence layer, delivered as a web map and a PDF.
**Roadmap.** Label more storms and retrain; add a rainfall-conditioned model once enough events exist; Alexandria urban-flood susceptibility; Lake Nasser water-extent trend; hyperspectral (Tanager/EnMAP) to separate wet soil, turbid water and sand where scenes exist; an automated pipeline triggered by rainfall alerts.

*Data: Copernicus Sentinel-1/2 and DEM (ESA/EU), ESA WorldCover, OpenStreetMap contributors, Open-Meteo, via Microsoft Planetary Computer.*